#NB07 — Governed QuantConnect / LEAN Implementation Bridge

1. Across NB00–NB10, we progressively built a complete quantitative research institution, beginning with controlled data and moving through features, multiple predictive models, portfolio construction, execution realism, independent risk, governance, agents, and closed-loop research orchestration.  
2. By NB10, the system could accept an investment objective written in natural language, use an LLM to interpret that objective and propose relevant research paths, and then use deterministic code to validate those proposals, run competing models and portfolio alternatives, apply stress and falsification tests, and reject weaker candidates.  
3. The result of NB10 was therefore not an automatically deployed trading strategy, but a **governed research design** describing the models, signals, portfolio rules, risk controls, and execution assumptions that best survived the research process under the stated objective.  
4. NB07 then translated that NB10-style research architecture into QuantConnect/LEAN, preserving the same logic of model competition, deterministic champion selection, portfolio construction, corporate-action handling, independent risk, and audit rather than simply hard-coding a single frozen model.  
5. In short, NB10 produced the governed research design from a natural-language objective, and NB07 operationalized that design as an executable, research-only QuantConnect implementation.




## From an autonomous quantitative research institution to a platform-executable research system

NB07 occupies a deliberately difficult position in the architecture of this course. Everything before it has taught us how to construct a quantitative research institution: how to create a controlled market substrate; how to engineer point-in-time features; how to compare heterogeneous models rather than fetishize one algorithm; how to transform predictions into strategies and strategies into portfolios; how to represent costs, execution, risk, governance, provenance, agents, constellations, budgets, state transitions, recovery, falsification, and human authority; and, in the later integrated notebooks, how to place a real LLM inside a bounded cognitive control plane without allowing language generation to become execution authority. NB07 asks a different question. It asks what happens when this institution must leave the protected notebook laboratory and express a research strategy inside an actual event-driven algorithmic-trading engine.

That transition is not a matter of copying a few equations into a new API. A research notebook and an execution engine inhabit different semantic worlds. In a notebook, the researcher frequently thinks in matrices and completed histories: a DataFrame exists, features are calculated, a model is fit, predictions are produced, a backtest consumes those predictions, and evaluation follows. An event-driven engine such as LEAN experiences the world sequentially. Time advances through data slices. Corporate actions arrive as events. Securities become active or inactive. Training may be scheduled. Orders have lifecycles. Fees and slippage are produced by reality models. Portfolio state changes because fills occur, not because a vector of desired positions was written into a table. The implementation therefore has to preserve the *meaning* of the research process while changing its computational form.

The central pedagogical idea of NB07 is **semantic compilation**. The high-level research system produces an intermediate representation—a governed `StrategySpec`—that describes what is to be tested: universe, data semantics, features, model families, training schedule, signal transformation, portfolio construction, cost assumptions, risk limits, corporate-action handling, audit requirements, and explicit prohibitions. That specification is not executable merely because it is machine readable. A deterministic compiler checks the specification, rejects prohibited or incomplete states, maps approved concepts to QuantConnect/LEAN constructs, and produces a project whose execution semantics can be inspected and tested. This is exactly the same architectural principle used in NB09 and NB10 when an LLM proposal is treated as an intermediate representation rather than as an instruction to execute. Here the compiler target is no longer a generic research DAG; it is a LEAN-compatible Python project.

The resulting notebook has two planes. The **control plane** remains notebook-side. It may receive an ordinary-language objective, optionally ask an LLM to propose a bounded research plan, validate that plan, compile it into a `StrategySpec`, calculate hashes, construct an experiment record, and generate code. The **execution plane** is QuantConnect/LEAN. It receives only approved deterministic artifacts. The LEAN algorithm never asks an LLM what to trade. It does not accept a model-generated order instruction. It does not enlarge its universe or permissions on its own. It executes the code and configuration produced by the compiler under a research-only constitution.

This distinction matters because an LLM and a trading engine solve fundamentally different problems. The LLM is useful for semantic interpretation, hypothesis expansion, experiment design, and research critique. LEAN is useful for deterministic event sequencing, data access, corporate-action processing, order simulation, portfolio accounting, reality modeling, and backtesting. Trying to make either component perform the other's function weakens the system. NB07 therefore keeps cognition and execution heterogeneous by design.

The QuantConnect implementation also forces us to confront data realism. NB00 created a canonical synthetic SQLite world with thirty equities, six sectors, regimes, corporate actions, controlled defects, manifests, and provenance. QuantConnect supplies a different substrate. Its U.S. equity data and Security Master support actual market history and corporate actions; its `QCAlgorithm` runtime calls `initialize` once, advances through time, delivers data through event handlers, supports historical requests, and can apply fee and slippage models. The correct bridge is therefore not to pretend that the NB00 SQLite database *is* QuantConnect data. Instead, NB07 maps the **contracts** created in NB00 onto QuantConnect-native data semantics. The default project uses a fixed thirty-security, six-group U.S. equity research universe so that the shape of the experiment remains close to NB10. The universe is deliberately configurable, and no claim is made that the chosen names constitute an investable portfolio or a survivorship-bias-free institutional universe. For production-grade research, universe selection must itself be point-in-time and independently validated.

Corporate actions receive special treatment. They were explicit records in NB00 and became event-aware inputs in NB10. In LEAN, dividends and splits are first-class events. NB07 therefore records them, incorporates a conservative event cool-down into portfolio eligibility, and prevents future corporate-action knowledge from becoming a predictive feature. Price normalization is declared explicitly in the `StrategySpec`, because a change in normalization changes the meaning of historical prices and therefore the meaning of the feature set.

The model layer likewise mirrors the integrated research architecture. The generated QuantConnect algorithm can train a bounded library containing logistic regression, k-nearest neighbors, random forest, multilayer perceptron, and linear regression. Training occurs through LEAN's training mechanism on historical observations available at the training time. Features are constructed from trailing information only. Candidate models are evaluated on a chronological internal validation segment. A deterministic selector chooses the current research champion using predeclared metrics and tie-breakers. The champion is therefore selected mathematically, not rhetorically. An LLM may have proposed which model families should be examined, but it cannot choose the winner after seeing the results.

Signal extraction is separated from portfolio construction. A model produces a score or probability; a deterministic strategy layer transforms that output into cross-sectional ranks; a portfolio layer converts ranks into weights; and an independent risk governor clamps or rejects those weights before they reach the order API. This layering is important. It lets us ask whether a failure belongs to the model, the signal rule, the portfolio method, execution costs, or risk constraints. It also makes the implementation falsifiable.

The risk layer is intentionally independent in logic even though this teaching implementation runs inside a single `QCAlgorithm` process. The `IndependentRiskGovernor` receives candidate weights and current portfolio state but does not participate in model fitting. It enforces maximum position size, maximum gross exposure, sector concentration, and portfolio drawdown rules. A production institution would separate this function more strongly—possibly into a distinct service, account-level control, or broker-side limit—but the notebook preserves the conceptual separation required by NB02, NB09, and NB10.

Reality modeling is also explicit. The QuantConnect project attaches fee and slippage models to each subscribed security. The goal is not to claim that any one calibration is universally realistic; it is to make costs visible, configurable, and testable rather than implicit. The falsification layer checks that costs cannot simply disappear from the implementation. The experiment registry records which assumptions produced which code hash so that results can be reproduced and challenged.

NB07 also preserves the course's doctrine of authority. **Backtesting is permitted; live trading is not.** The generated algorithm contains a hard live-mode denial. The compiled `StrategySpec` carries `deployment_authorized = false` and `live_trading_authorized = false`. The validator rejects any mission requesting broker connectivity, live orders, deployment, or self-promotion. The project is therefore implementation-capable without being self-authorizing.

This is a crucial distinction for teaching autonomous systems. A system becomes more useful when it can cross interfaces, call tools, create platform artifacts, and execute a closed-loop experiment. It becomes more dangerous if those capabilities silently expand its mandate. The correct response is not to avoid implementation; it is to implement **under explicit authority boundaries**. NB07 is therefore the notebook in which the course learns how to cross the laboratory boundary without erasing governance.

Finally, this notebook is deliberately dual-mode. In Colab or a local Jupyter environment it acts as a compiler, validator, code generator, and semantic mirror. In a QuantConnect Research environment, the same notebook can be used to inspect platform data through `QuantBook`. The generated `main.py` and `strategy_config.py` form the backtest project. This separation means the pedagogical narrative remains visible while the actual LEAN algorithm remains compact enough to audit.

> **NB07 doctrine:** research intent may be expressed at a high level; an LLM may help interpret it; a deterministic compiler translates it; QuantConnect executes only the compiled research artifact; independent risk constrains it; provenance records it; and humans retain promotion and deployment authority.

**Research and education only. No live-trading authorization. No brokerage authorization. No self-promotion.**

## 1. Architectural mapping: NB10 concepts → QuantConnect / LEAN constructs

The purpose of this mapping is not to claim that two systems are identical. It is to make the translation explicit enough that disagreements can be found. QuantConnect's current Python workflow uses `QCAlgorithm.initialize` as the one-time initialization entry point, `add_equity` and universe APIs for subscriptions, `history` for trailing data, Scheduled Events and `train` for timed work, corporate-action handlers for dividends and splits, reality models for fees/slippage, and the Object Store for persistent artifacts. The bridge below maps our institutional concepts onto those platform mechanisms.

| Institutional concept | NB10 research meaning | NB07 / LEAN implementation |
|---|---|---|
| Mission | Natural-language bounded research objective | Notebook-side mission object |
| Hard gate | Scope, permission, budget, live-action denial | Deterministic pre-compilation validator |
| LLM plan | Intermediate research proposal | Optional notebook-side proposal only |
| Plan compiler | Proposal → executable DAG | Proposal → `StrategySpec` → project files |
| Canonical data contract | Point-in-time prices, volume, events, metadata | QuantConnect subscriptions + history + Security Master events |
| Feature engine | Trailing, point-in-time features | Historical window computed at training/rebalance time |
| Model library | Heterogeneous candidate models | sklearn models trained through LEAN `train` |
| Champion | Deterministic winner | Chronological validation metric + tie-breakers |
| Strategy | Model/rule output → positions | Cross-sectional ranking rule |
| Portfolio | Equal weight / inverse vol / risk methods | Configurable target-weight compiler |
| Execution | Cost-aware simulated orders | LEAN orders + fee/slippage models |
| Independent risk | Separate challenge layer | `IndependentRiskGovernor` before order submission |
| Corporate events | Event-aware, no look-ahead | `on_dividends` / `on_splits` + cool-down |
| Audit | Hash-linked research evidence | Code/spec hashes + event records + Object Store artifact |
| Promotion | Human-only | Always `HUMAN_REVIEW_REQUIRED` |
| Live authority | Denied | `live_mode` hard fail |

The current QuantConnect documentation confirms these event-driven and persistence mechanisms; NB07 keeps the implementation narrow enough that a student can inspect every mapping rather than hiding the translation behind a large framework.

## 2. Environment and operating modes

NB07 can be opened in Colab, locally, or in QuantConnect Research. The compiler itself uses ordinary Python. QuantConnect-specific classes are used only by the generated project or when `QuantBook` is available.

The notebook does **not** require QuantConnect credentials to generate the project. Platform backtest results are deliberately not fabricated in Colab.

In [ ]:
from __future__ import annotations

import ast
import hashlib
import json
import os
import pathlib
import textwrap
from copy import deepcopy
from datetime import datetime

NB07_ROOT = pathlib.Path("/content/nb07_quantconnect_bridge") if pathlib.Path("/content").exists() else pathlib.Path("./nb07_quantconnect_bridge")
NB07_ROOT.mkdir(parents=True, exist_ok=True)

try:
    from AlgorithmImports import QuantBook, Resolution
    QUANTCONNECT_RESEARCH_AVAILABLE = True
except Exception:
    QUANTCONNECT_RESEARCH_AVAILABLE = False

print("NB07 root:", NB07_ROOT)
print("QuantConnect Research available:", QUANTCONNECT_RESEARCH_AVAILABLE)

NB07 root: /content/nb07_quantconnect_bridge
QuantConnect Research available: False


## 3. The expanded `StrategySpec`: the contract between research and LEAN

The original bridge used a minimum description with twenty-two required fields. This rewrite preserves all of those ideas and expands the contract so that the translation can carry the additional assumptions introduced in NB08–NB10. The specification is intentionally more verbose than a typical configuration file because silent defaults are precisely what a governed bridge should avoid.

A `StrategySpec` does not authorize execution. It is a typed research artifact. The compiler must still validate it before generating platform code.

In [ ]:
REQUIRED_STRATEGY_FIELDS = [
    "strategy_id",
    "version",
    "research_question",
    "asset_class",
    "universe",
    "universe_groups",
    "universe_mode",
    "resolution",
    "calendar",
    "timezone",
    "data_normalization",
    "feature_contract",
    "label_horizon",
    "model_families",
    "champion_selection_rule",
    "training_lookback_bars",
    "training_schedule",
    "signal_rule",
    "position_rule",
    "portfolio_method",
    "rebalance_rule",
    "cost_model",
    "slippage_model",
    "risk_limits",
    "corporate_action_policy",
    "start_date",
    "end_date",
    "starting_cash",
    "benchmark",
    "warmup_bars",
    "seed",
    "expected_outputs",
    "known_limitations",
    "research_only",
    "deployment_authorized",
    "live_trading_authorized",
    "promotion_view",
]

APPROVED_MODELS = [
    "logistic_regression",
    "knn",
    "random_forest",
    "mlp",
    "linear_regression",
]

APPROVED_PORTFOLIOS = [
    "equal_weight",
    "inverse_volatility",
    "risk_parity_proxy",
    "volatility_target",
]

APPROVED_FEATURES = [
    "ret_1",
    "mom_5",
    "mom_20",
    "vol_20",
    "vol_60",
    "volume_z20",
    "dollar_volume_z20",
    "ma_ratio_20",
    "ma_ratio_50",
]

UNIVERSE_BY_SECTOR = {
    "Technology": ["AAPL", "MSFT", "NVDA", "AVGO", "ORCL"],
    "Financials": ["JPM", "BAC", "GS", "MS", "C"],
    "Healthcare": ["JNJ", "UNH", "LLY", "MRK", "ABBV"],
    "Industrials": ["CAT", "GE", "HON", "UPS", "RTX"],
    "Consumer": ["AMZN", "WMT", "COST", "MCD", "HD"],
    "Energy": ["XOM", "CVX", "COP", "SLB", "EOG"],
}

DEFAULT_UNIVERSE = [ticker for names in UNIVERSE_BY_SECTOR.values() for ticker in names]

assert len(DEFAULT_UNIVERSE) == 30
assert len(set(DEFAULT_UNIVERSE)) == 30
print("StrategySpec contract loaded:", len(REQUIRED_STRATEGY_FIELDS), "required fields")

StrategySpec contract loaded: 37 required fields


## 4. Mission input and the hard gate

The user may change the `objective` while leaving the institutional boundary unchanged. As in NB10, the system distinguishes *what the researcher wants to investigate* from *what the system is allowed to do*. Backtesting a proposed strategy is inside scope. Connecting a broker, trading live, deploying automatically, or promoting the strategy without human review is outside scope and must fail before code generation.

In [ ]:
MISSION = {
    "objective": (
        "Evaluate whether a cross-sectional, event-aware equity strategy can produce "
        "robust risk-adjusted performance across sectors. Compare approved machine-learning "
        "families, use point-in-time trailing features, realistic costs, independent risk, "
        "corporate-action awareness, and falsification tests. Translate the resulting research "
        "plan into a QuantConnect/LEAN backtest project."
    ),
    "asset_class": "US_EQUITIES",
    "universe": "NB07_30_EQUITY_TEACHING_FIXTURE",
    "start_date": "2021-01-01",
    "end_date": "2025-12-31",
    "research_only": True,
    "live_trading_authorized": False,
    "deployment_authorized": False,
    "promotion_view": "HUMAN_REVIEW_REQUIRED",
}

PROHIBITED_TERMS = [
    "live trade",
    "live order",
    "connect broker",
    "brokerage connection",
    "deploy live",
    "production deployment",
    "self approve",
    "self-approve",
]

def hard_gate_mission(mission):
    required = [
        "objective", "asset_class", "universe", "start_date", "end_date",
        "research_only", "live_trading_authorized",
        "deployment_authorized", "promotion_view",
    ]
    missing = [field for field in required if field not in mission]
    if missing:
        return {"passed": False, "reason": f"missing_fields:{missing}"}

    objective = mission["objective"].lower()
    if any(term in objective for term in PROHIBITED_TERMS):
        return {"passed": False, "reason": "prohibited_external_effect"}

    if mission["asset_class"] != "US_EQUITIES":
        return {"passed": False, "reason": "unsupported_asset_class"}

    if mission["research_only"] is not True:
        return {"passed": False, "reason": "research_only_required"}

    if mission["live_trading_authorized"] is not False:
        return {"passed": False, "reason": "live_authority_must_be_false"}

    if mission["deployment_authorized"] is not False:
        return {"passed": False, "reason": "deployment_authority_must_be_false"}

    if mission["promotion_view"] != "HUMAN_REVIEW_REQUIRED":
        return {"passed": False, "reason": "human_review_required"}

    return {"passed": True, "reason": "ADMITTED_RESEARCH_ONLY"}

MISSION_GATE = hard_gate_mission(MISSION)
print(MISSION_GATE)
assert MISSION_GATE["passed"]

{'passed': True, 'reason': 'ADMITTED_RESEARCH_ONLY'}


## 5. Planner: NB10-like intent, without giving the LLM execution authority

NB07 can consume a plan imported from NB10, a deterministic reference plan, or—optionally—a real OpenAI planner. Regardless of origin, the proposal remains an intermediate representation. The compiler validates model names, portfolio methods, scope, resources, and authority before producing platform code.

The deterministic reference planner below makes the notebook runnable without an API key. It is not presented as a substitute for the NB10 LLM; it is a reproducible fixture for platform translation.

In [ ]:
def deterministic_reference_plan(mission):
    objective = mission["objective"].lower()

    models = list(APPROVED_MODELS)
    if "knn only" in objective:
        models = ["knn"]
    elif "linear only" in objective:
        models = ["linear_regression"]

    portfolio_methods = [
        "equal_weight",
        "inverse_volatility",
        "volatility_target",
    ]
    if "risk parity" in objective:
        portfolio_methods.append("risk_parity_proxy")

    return {
        "objective_interpretation": mission["objective"],
        "selected_model_tools": models,
        "selected_portfolio_methods": portfolio_methods,
        "selected_features": list(APPROVED_FEATURES),
        "corporate_action_policy": "observe events; two-day eligibility cool-down; never use future event knowledge",
        "champion_selection_rule": (
            "highest chronological validation balanced accuracy; "
            "tie-break by predefined complexity rank"
        ),
        "required_tests": [
            "chronological_split",
            "cost_sensitivity",
            "regime_stability",
            "parameter_sensitivity",
            "corporate_action_handling",
            "drawdown_limit",
            "live_mode_denial",
            "code_reproducibility",
        ],
        "promotion_view": "HUMAN_REVIEW_REQUIRED",
    }

PLAN_PROPOSAL = deterministic_reference_plan(MISSION)
print(json.dumps(PLAN_PROPOSAL, indent=2))

{
  "objective_interpretation": "Evaluate whether a cross-sectional, event-aware equity strategy can produce robust risk-adjusted performance across sectors. Compare approved machine-learning families, use point-in-time trailing features, realistic costs, independent risk, corporate-action awareness, and falsification tests. Translate the resulting research plan into a QuantConnect/LEAN backtest project.",
  "selected_model_tools": [
    "logistic_regression",
    "knn",
    "random_forest",
    "mlp",
    "linear_regression"
  ],
  "selected_portfolio_methods": [
    "equal_weight",
    "inverse_volatility",
    "volatility_target"
  ],
  "selected_features": [
    "ret_1",
    "mom_5",
    "mom_20",
    "vol_20",
    "vol_60",
    "volume_z20",
    "dollar_volume_z20",
    "ma_ratio_20",
    "ma_ratio_50"
  ],
  "corporate_action_policy": "observe events; two-day eligibility cool-down; never use future event knowledge",
  "champion_selection_rule": "highest chronological validation b

### Optional real LLM planner

If you want NB07 to make the same cognitive move as NB10, set `USE_OPENAI_PLANNER = True` and provide `OPENAI_API_KEY`. The schema is intentionally conservative: shape and enums are constrained remotely; semantic constraints such as uniqueness, authority, budgets, and cross-field logic are validated deterministically after parsing. This avoids the recurring error caused by unsupported JSON Schema keywords such as `uniqueItems`.

In [ ]:
USE_OPENAI_PLANNER = False
OPENAI_MODEL = "gpt-5.6-terra"

def planner_json_schema():
    return {
        "type": "object",
        "properties": {
            "objective_interpretation": {"type": "string"},
            "selected_model_tools": {
                "type": "array",
                "items": {"type": "string", "enum": APPROVED_MODELS},
            },
            "selected_portfolio_methods": {
                "type": "array",
                "items": {"type": "string", "enum": APPROVED_PORTFOLIOS},
            },
            "selected_features": {
                "type": "array",
                "items": {"type": "string", "enum": APPROVED_FEATURES},
            },
            "corporate_action_policy": {"type": "string"},
            "champion_selection_rule": {"type": "string"},
            "required_tests": {
                "type": "array",
                "items": {"type": "string"},
            },
            "promotion_view": {
                "type": "string",
                "enum": ["HUMAN_REVIEW_REQUIRED"],
            },
        },
        "required": [
            "objective_interpretation",
            "selected_model_tools",
            "selected_portfolio_methods",
            "selected_features",
            "corporate_action_policy",
            "champion_selection_rule",
            "required_tests",
            "promotion_view",
        ],
        "additionalProperties": False,
    }

def _dedupe(values):
    return list(dict.fromkeys(values))

def normalize_plan(plan):
    out = deepcopy(plan)
    for field in [
        "selected_model_tools",
        "selected_portfolio_methods",
        "selected_features",
        "required_tests",
    ]:
        out[field] = _dedupe(out.get(field, []))
    return out

def call_openai_planner(mission):
    if not USE_OPENAI_PLANNER:
        return deterministic_reference_plan(mission)

    from openai import OpenAI

    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    response = client.responses.create(
        model=OPENAI_MODEL,
        instructions=(
            "You are the bounded research planner for NB07. "
            "Propose only QuantConnect research/backtest experiments. "
            "Never authorize live trading, deployment, brokerage connectivity, or self-promotion. "
            "Use only the model, feature, and portfolio enums supplied by the schema."
        ),
        input=json.dumps(mission, sort_keys=True),
        reasoning={"effort": "medium"},
        text={
            "format": {
                "type": "json_schema",
                "name": "nb07_quantconnect_research_plan",
                "schema": planner_json_schema(),
                "strict": True,
            }
        },
    )
    return normalize_plan(json.loads(response.output_text))

if USE_OPENAI_PLANNER:
    PLAN_PROPOSAL = call_openai_planner(MISSION)
    print(json.dumps(PLAN_PROPOSAL, indent=2))

## 6. Deterministic compiler: proposal → `StrategySpec`

This is the point at which NB07 becomes an implementation bridge rather than a code-generation demo. The compiler decides what the platform is actually allowed to receive. Unknown models are rejected. Unsupported portfolio methods are rejected. The human-review requirement is immutable. Live and deployment authority remain false. The compiler also resolves ambiguities that an LLM is not permitted to resolve on its own—for example, which portfolio method becomes the default implementation in `main.py`.

In [ ]:
def stable_hash(value):
    payload = json.dumps(value, sort_keys=True, default=str, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def validate_plan(plan):
    if not plan["selected_model_tools"]:
        raise ValueError("At least one approved model family is required.")
    unknown_models = sorted(set(plan["selected_model_tools"]) - set(APPROVED_MODELS))
    if unknown_models:
        raise ValueError(f"Unknown models: {unknown_models}")

    unknown_portfolios = sorted(set(plan["selected_portfolio_methods"]) - set(APPROVED_PORTFOLIOS))
    if unknown_portfolios:
        raise ValueError(f"Unknown portfolio methods: {unknown_portfolios}")

    unknown_features = sorted(set(plan["selected_features"]) - set(APPROVED_FEATURES))
    if unknown_features:
        raise ValueError(f"Unknown features: {unknown_features}")

    if plan["promotion_view"] != "HUMAN_REVIEW_REQUIRED":
        raise ValueError("Promotion authority cannot be delegated to the planner.")

    return True

def compile_strategy_spec(mission, plan):
    validate_plan(plan)

    # The generated implementation uses one declared portfolio method.
    preferred = ["inverse_volatility", "volatility_target", "equal_weight", "risk_parity_proxy"]
    portfolio_method = next(
        (method for method in preferred if method in plan["selected_portfolio_methods"]),
        plan["selected_portfolio_methods"][0],
    )

    spec = {
        "strategy_id": "NB07_QC_GOVERNED_MULTI_MODEL_V2",
        "version": "2.0.0-research",
        "research_question": mission["objective"],
        "asset_class": "US_EQUITIES",
        "universe": list(DEFAULT_UNIVERSE),
        "universe_groups": deepcopy(UNIVERSE_BY_SECTOR),
        "universe_mode": "STATIC_TEACHING_FIXTURE",
        "resolution": "DAILY",
        "calendar": "US_EQUITY",
        "timezone": "America/New_York",
        "data_normalization": "ADJUSTED",
        "feature_contract": list(plan["selected_features"]),
        "label_horizon": 1,
        "model_families": list(plan["selected_model_tools"]),
        "champion_selection_rule": plan["champion_selection_rule"],
        "training_lookback_bars": 320,
        "training_schedule": "MONTH_START_BEFORE_OPEN",
        "signal_rule": "cross-sectional ranking of champion model score",
        "position_rule": "top 20 percent long; bottom 20 percent short; event-cooldown exclusions",
        "portfolio_method": portfolio_method,
        "rebalance_rule": "MONTH_START_AFTER_OPEN",
        "cost_model": {"type": "ConstantFeeModel", "fee_usd_per_order": 0.0},
        "slippage_model": {
            "type": "VolumeShareSlippageModel",
            "volume_limit": 0.025,
            "price_impact": 0.10,
        },
        "risk_limits": {
            "max_gross": 1.0,
            "max_position_weight": 0.10,
            "max_sector_gross": 0.35,
            "max_portfolio_drawdown": 0.12,
        },
        "corporate_action_policy": plan["corporate_action_policy"],
        "start_date": mission["start_date"],
        "end_date": mission["end_date"],
        "starting_cash": 1_000_000,
        "benchmark": "SPY",
        "warmup_bars": 80,
        "seed": 20260906,
        "expected_outputs": [
            "model_diagnostics",
            "champion",
            "orders",
            "fills",
            "portfolio_state",
            "risk_report",
            "corporate_action_events",
            "audit_bundle",
        ],
        "known_limitations": [
            "static teaching universe may contain survivorship bias",
            "single-process risk separation is logical rather than service-level",
            "QuantConnect cloud compilation and matched-data results must be run on platform",
            "backtest results do not authorize deployment or live trading",
        ],
        "research_only": True,
        "deployment_authorized": False,
        "live_trading_authorized": False,
        "promotion_view": "HUMAN_REVIEW_REQUIRED",
    }

    missing = [field for field in REQUIRED_STRATEGY_FIELDS if field not in spec]
    if missing:
        raise ValueError(f"Compiled StrategySpec missing fields: {missing}")

    return spec

STRATEGY_SPEC = compile_strategy_spec(MISSION, PLAN_PROPOSAL)
STRATEGY_SPEC_HASH = stable_hash(STRATEGY_SPEC)

print("StrategySpec hash:", STRATEGY_SPEC_HASH)
print("models:", STRATEGY_SPEC["model_families"])
print("portfolio:", STRATEGY_SPEC["portfolio_method"])

StrategySpec hash: 36211d0b8001423cb79b65dfd3692fabbcc1e629022097b3ffbdf52c28ab0069
models: ['logistic_regression', 'knn', 'random_forest', 'mlp', 'linear_regression']
portfolio: inverse_volatility


## 7. QuantConnect-native data semantics and corporate actions

The implementation uses a fixed thirty-equity research fixture grouped into six broad sectors so that the scale resembles NB10. The fixture is convenient for teaching and code validation, but it is **not** a point-in-time survivorship-free universe. That limitation is explicit in the specification.

Within LEAN, the feature engine requests only trailing history available at the current algorithm time. Dividends and splits are received as platform events and recorded. They create a short eligibility cool-down; future events are never passed to the model.

The generated project uses adjusted U.S. equity data for feature continuity and declares that choice in the `StrategySpec`. If an institutional implementation requires raw or total-return semantics, that is a controlled specification change, not a silent code edit.

## 8. Model laboratory and mathematical champion selection

The generated LEAN project trains the same core heterogeneous families used in the integrated research system: logistic regression, KNN, random forest, a bounded MLP, and linear regression. The training dataset pools trailing observations across securities and preserves chronological order. The most recent portion is held back as an internal validation segment. Each model produces an up/down classification, directly or through the sign of a regression forecast.

Champion selection is deterministic. The primary criterion is validation balanced accuracy; the tie-breaker is a predeclared complexity order. An LLM may recommend which families should be included in the experiment, but once empirical evaluation begins, rhetoric has no role in choosing the champion.

## 9. Signal, portfolio, execution, and independent risk

The champion produces a cross-sectional score. Scores are ranked, the strongest fraction becomes the long candidate set, and the weakest fraction becomes the short candidate set. Portfolio construction is then a separate step. The default implementation uses inverse-volatility weights, although equal weighting and volatility targeting remain permitted compiled choices.

Before any target reaches `set_holdings`, an `IndependentRiskGovernor` receives the candidate weights. It can clamp security weights, reduce sector concentration, enforce gross exposure, or halt the portfolio after a drawdown breach. This risk component does not train models and does not choose the champion.

Execution is performed only by LEAN. Each security receives explicit fee and slippage reality models. Orders, fills, risk decisions, champion changes, dividends, and splits are recorded in the audit stream.

## 10. Generated QuantConnect project

The next cell writes the deterministic `strategy_config.py` and `main.py`. The LLM never writes orders directly. The compiler materializes a static project from the validated specification.

In [ ]:
STRATEGY_CONFIG_SOURCE = """# Auto-generated by NB07. Research/backtest only.

SEED = 20260906
RESEARCH_ONLY = True
DEPLOYMENT_AUTHORIZED = False
LIVE_TRADING_AUTHORIZED = False

START_DATE = (2021, 1, 1)
END_DATE = (2025, 12, 31)
STARTING_CASH = 1_000_000
BENCHMARK = "SPY"

RESOLUTION = "DAILY"
DATA_NORMALIZATION = "ADJUSTED"
WARMUP_BARS = 80
TRAINING_LOOKBACK_BARS = 320
RETRAIN_FREQUENCY = "MONTHLY"
REBALANCE_FREQUENCY = "MONTHLY"

FEATURES = [
    "ret_1",
    "mom_5",
    "mom_20",
    "vol_20",
    "vol_60",
    "volume_z20",
    "dollar_volume_z20",
    "ma_ratio_20",
    "ma_ratio_50",
]

MODEL_FAMILIES = [
    "logistic_regression",
    "knn",
    "random_forest",
    "mlp",
    "linear_regression",
]
CHAMPION_SELECTION_RULE = (
    "highest chronological validation balanced accuracy; "
    "tie-break by predefined model-complexity order"
)

SIGNAL_MODE = "CROSS_SECTIONAL_LONG_SHORT"
LONG_FRACTION = 0.20
SHORT_FRACTION = 0.20
MIN_SIGNAL_ABS = 0.00

PORTFOLIO_METHOD = "inverse_volatility"
TARGET_DAILY_VOL = 0.010
MAX_GROSS = 1.00
MAX_POSITION_WEIGHT = 0.10
MAX_SECTOR_GROSS = 0.35
MAX_PORTFOLIO_DRAWDOWN = 0.12
EVENT_COOLDOWN_DAYS = 2

PER_ORDER_FEE_USD = 0.00
VOLUME_SHARE_LIMIT = 0.025
VOLUME_SHARE_PRICE_IMPACT = 0.10

UNIVERSE_BY_SECTOR = {
    "Technology": ["AAPL", "MSFT", "NVDA", "AVGO", "ORCL"],
    "Financials": ["JPM", "BAC", "GS", "MS", "C"],
    "Healthcare": ["JNJ", "UNH", "LLY", "MRK", "ABBV"],
    "Industrials": ["CAT", "GE", "HON", "UPS", "RTX"],
    "Consumer": ["AMZN", "WMT", "COST", "MCD", "HD"],
    "Energy": ["XOM", "CVX", "COP", "SLB", "EOG"],
}

AUDIT_OBJECT_STORE_KEY = "NB07/audit_bundle.json"
"""
MAIN_SOURCE = """from AlgorithmImports import *
from strategy_config import *

import json
import hashlib
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import balanced_accuracy_score


def stable_hash(value):
    payload = json.dumps(value, sort_keys=True, default=str, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


class IndependentRiskGovernor:
    \"\"\"
    Logically independent risk review.
    It does not fit models and does not choose the research champion.
    It receives candidate weights and can clamp or halt them.
    \"\"\"

    def __init__(self):
        self.peak_value = None
        self.halted = False
        self.last_report = {}

    def review(self, algorithm, candidate_weights):
        value = float(algorithm.portfolio.total_portfolio_value)

        if self.peak_value is None:
            self.peak_value = value
        self.peak_value = max(self.peak_value, value)

        drawdown = 0.0 if self.peak_value <= 0 else value / self.peak_value - 1.0
        if drawdown <= -MAX_PORTFOLIO_DRAWDOWN:
            self.halted = True

        if self.halted:
            self.last_report = {
                "status": "HALT",
                "portfolio_value": value,
                "drawdown": drawdown,
                "reason": "portfolio_drawdown_limit",
            }
            return {}

        bounded = {}
        for symbol, weight in candidate_weights.items():
            bounded[symbol] = float(
                max(-MAX_POSITION_WEIGHT, min(MAX_POSITION_WEIGHT, weight))
            )

        # Sector concentration control.
        for sector, symbols in algorithm._symbols_by_sector.items():
            sector_symbols = [s for s in symbols if s in bounded]
            sector_gross = sum(abs(bounded[s]) for s in sector_symbols)
            if sector_gross > MAX_SECTOR_GROSS and sector_gross > 0:
                scale = MAX_SECTOR_GROSS / sector_gross
                for s in sector_symbols:
                    bounded[s] *= scale

        gross = sum(abs(w) for w in bounded.values())
        if gross > MAX_GROSS and gross > 0:
            scale = MAX_GROSS / gross
            bounded = {s: w * scale for s, w in bounded.items()}

        self.last_report = {
            "status": "PASS",
            "portfolio_value": value,
            "drawdown": drawdown,
            "gross": sum(abs(w) for w in bounded.values()),
            "net": sum(bounded.values()),
            "positions": len([w for w in bounded.values() if abs(w) > 1e-12]),
        }
        return bounded


class GovernedQuantConnectBridge(QCAlgorithm):
    \"\"\"
    NB07 QuantConnect/LEAN bridge.

    The algorithm is intentionally research-only. The LLM/control plane is external
    to this runtime. LEAN receives only deterministic configuration and code.
    \"\"\"

    def initialize(self):
        if self.live_mode or LIVE_TRADING_AUTHORIZED or DEPLOYMENT_AUTHORIZED:
            raise RuntimeError(
                "NB07 governance denial: live trading/deployment is not authorized."
            )

        self.set_start_date(*START_DATE)
        self.set_end_date(*END_DATE)
        self.set_cash(STARTING_CASH)

        self._audit = []
        self._event_dates = {}
        self._model_diagnostics = []
        self._champion_name = None
        self._champion_model = None
        self._champion_kind = None
        self._risk = IndependentRiskGovernor()

        self._benchmark_symbol = self.add_equity(
            BENCHMARK,
            Resolution.DAILY,
            data_normalization_mode=DataNormalizationMode.ADJUSTED
        ).symbol
        self.set_benchmark(self._benchmark_symbol)

        self._symbols = []
        self._sector_by_symbol = {}
        self._symbols_by_sector = {}

        for sector, tickers in UNIVERSE_BY_SECTOR.items():
            sector_symbols = []
            for ticker in tickers:
                security = self.add_equity(
                    ticker,
                    Resolution.DAILY,
                    data_normalization_mode=DataNormalizationMode.ADJUSTED
                )
                security.set_fee_model(ConstantFeeModel(PER_ORDER_FEE_USD))
                security.set_slippage_model(
                    VolumeShareSlippageModel(
                        VOLUME_SHARE_LIMIT,
                        VOLUME_SHARE_PRICE_IMPACT
                    )
                )
                symbol = security.symbol
                self._symbols.append(symbol)
                sector_symbols.append(symbol)
                self._sector_by_symbol[symbol] = sector
                self._event_dates[symbol] = {
                    "dividend": None,
                    "split": None,
                }
            self._symbols_by_sector[sector] = sector_symbols

        self.set_warm_up(WARMUP_BARS, Resolution.DAILY)

        # Model training runs inside LEAN's training mechanism.
        self.train(
            self.date_rules.month_start(self._benchmark_symbol),
            self.time_rules.before_market_open(self._benchmark_symbol, 60),
            self._train_models,
        )

        # Portfolio rebalance follows training on the same monthly schedule.
        self.schedule.on(
            self.date_rules.month_start(self._benchmark_symbol),
            self.time_rules.after_market_open(self._benchmark_symbol, 30),
            self._rebalance,
        )

        self._record(
            "INITIALIZED",
            {
                "universe_size": len(self._symbols),
                "model_families": MODEL_FAMILIES,
                "portfolio_method": PORTFOLIO_METHOD,
                "research_only": RESEARCH_ONLY,
            },
        )

    def _record(self, event_type, payload):
        record = {
            "time": str(self.time),
            "event_type": event_type,
            "payload": payload,
        }
        record["hash"] = stable_hash(record)
        self._audit.append(record)

    def _build_model(self, name):
        if name == "logistic_regression":
            return (
                Pipeline([
                    ("scale", StandardScaler()),
                    ("model", LogisticRegression(
                        max_iter=500,
                        random_state=SEED,
                    )),
                ]),
                "classification",
            )
        if name == "knn":
            return (
                Pipeline([
                    ("scale", StandardScaler()),
                    ("model", KNeighborsClassifier(
                        n_neighbors=9,
                        weights="distance",
                    )),
                ]),
                "classification",
            )
        if name == "random_forest":
            return (
                RandomForestClassifier(
                    n_estimators=160,
                    max_depth=6,
                    min_samples_leaf=12,
                    random_state=SEED,
                    n_jobs=1,
                ),
                "classification",
            )
        if name == "mlp":
            return (
                Pipeline([
                    ("scale", StandardScaler()),
                    ("model", MLPClassifier(
                        hidden_layer_sizes=(24, 12),
                        max_iter=180,
                        early_stopping=True,
                        random_state=SEED,
                    )),
                ]),
                "classification",
            )
        if name == "linear_regression":
            return (
                Pipeline([
                    ("scale", StandardScaler()),
                    ("model", LinearRegression()),
                ]),
                "regression",
            )
        raise ValueError(f"Unknown approved model family: {name}")

    def _feature_frame_for_symbol(self, frame):
        frame = frame.sort_index().copy()
        close = frame["close"].astype(float)
        volume = frame["volume"].astype(float).replace(0, np.nan)
        dollar_volume = (close * volume).replace(0, np.nan)

        out = pd.DataFrame(index=frame.index)
        out["ret_1"] = close.pct_change(1)
        out["mom_5"] = close.pct_change(5)
        out["mom_20"] = close.pct_change(20)
        out["vol_20"] = close.pct_change().rolling(20).std()
        out["vol_60"] = close.pct_change().rolling(60).std()

        vol_mean = volume.rolling(20).mean()
        vol_std = volume.rolling(20).std().replace(0, np.nan)
        out["volume_z20"] = (volume - vol_mean) / vol_std

        dv_mean = dollar_volume.rolling(20).mean()
        dv_std = dollar_volume.rolling(20).std().replace(0, np.nan)
        out["dollar_volume_z20"] = (dollar_volume - dv_mean) / dv_std

        out["ma_ratio_20"] = close / close.rolling(20).mean() - 1.0
        out["ma_ratio_50"] = close / close.rolling(50).mean() - 1.0

        out["label_return_1"] = close.pct_change().shift(-1)
        out["label_up"] = (out["label_return_1"] > 0).astype(int)
        return out

    def _build_training_dataset(self):
        history = self.history(
            self._symbols,
            TRAINING_LOOKBACK_BARS,
            Resolution.DAILY,
        )
        if history is None or history.empty:
            return pd.DataFrame()

        parts = []
        for symbol in self._symbols:
            try:
                frame = history.loc[symbol]
            except Exception:
                continue

            if frame is None or len(frame) < 80:
                continue

            features = self._feature_frame_for_symbol(frame)
            features["symbol"] = str(symbol)
            features["event_time"] = features.index
            features = features.dropna(
                subset=FEATURES + ["label_return_1"]
            )
            parts.append(features)

        if not parts:
            return pd.DataFrame()

        dataset = pd.concat(parts, axis=0)
        return dataset.sort_values("event_time").reset_index(drop=True)

    def _train_models(self):
        if self.live_mode:
            raise RuntimeError("NB07 governance denial: live execution is prohibited.")
        if self.is_warming_up:
            return

        dataset = self._build_training_dataset()
        if dataset.empty or len(dataset) < 500:
            self._record("TRAIN_SKIPPED", {"rows": int(len(dataset))})
            return

        cut = int(len(dataset) * 0.80)
        train = dataset.iloc[:cut].copy()
        valid = dataset.iloc[cut:].copy()

        X_train = train[FEATURES].values
        X_valid = valid[FEATURES].values
        y_train_up = train["label_up"].astype(int).values
        y_valid_up = valid["label_up"].astype(int).values
        y_train_return = train["label_return_1"].astype(float).values

        diagnostics = []
        candidates = []

        for complexity, name in enumerate(MODEL_FAMILIES):
            model, kind = self._build_model(name)
            try:
                if kind == "classification":
                    model.fit(X_train, y_train_up)
                    pred_up = model.predict(X_valid).astype(int)
                else:
                    model.fit(X_train, y_train_return)
                    pred_return = model.predict(X_valid).astype(float)
                    pred_up = (pred_return > 0).astype(int)

                score = float(balanced_accuracy_score(y_valid_up, pred_up))
                diagnostics.append({
                    "model": name,
                    "kind": kind,
                    "validation_balanced_accuracy": score,
                    "complexity_rank": complexity,
                })
                candidates.append(
                    (score, -complexity, name, model, kind)
                )
            except Exception as exc:
                diagnostics.append({
                    "model": name,
                    "status": "FAILED",
                    "error": str(exc),
                    "complexity_rank": complexity,
                })

        if not candidates:
            self._record("TRAIN_FAILED", {"diagnostics": diagnostics})
            return

        # Deterministic champion selection.
        candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
        score, _, name, model, kind = candidates[0]

        self._champion_name = name
        self._champion_model = model
        self._champion_kind = kind
        self._model_diagnostics = diagnostics

        self._record(
            "CHAMPION_SELECTED",
            {
                "champion": name,
                "kind": kind,
                "validation_balanced_accuracy": score,
                "diagnostics": diagnostics,
            },
        )

    def _latest_features(self):
        history = self.history(
            self._symbols,
            max(80, WARMUP_BARS),
            Resolution.DAILY,
        )
        if history is None or history.empty:
            return pd.DataFrame()

        rows = []
        for symbol in self._symbols:
            try:
                frame = history.loc[symbol]
            except Exception:
                continue

            if frame is None or len(frame) < 65:
                continue

            features = self._feature_frame_for_symbol(frame)
            latest = features.iloc[-1]
            if latest[FEATURES].isna().any():
                continue

            row = latest[FEATURES].to_dict()
            row["symbol_object"] = symbol
            rows.append(row)

        return pd.DataFrame(rows)

    def _recent_corporate_action(self, symbol):
        dates = self._event_dates.get(symbol, {})
        for event_date in dates.values():
            if event_date is None:
                continue
            if (self.time.date() - event_date).days <= EVENT_COOLDOWN_DAYS:
                return True
        return False

    def _model_scores(self, latest):
        X = latest[FEATURES].values
        if self._champion_kind == "classification":
            if hasattr(self._champion_model, "predict_proba"):
                probability = self._champion_model.predict_proba(X)[:, 1]
                return probability - 0.5
            return self._champion_model.predict(X).astype(float) - 0.5
        return self._champion_model.predict(X).astype(float)

    def _portfolio_weights(self, latest):
        scores = self._model_scores(latest)
        records = []

        for i, row in latest.iterrows():
            symbol = row["symbol_object"]
            if self._recent_corporate_action(symbol):
                continue
            records.append({
                "symbol": symbol,
                "score": float(scores[i]),
                "vol": max(float(row["vol_20"]), 1e-6),
            })

        if len(records) < 6:
            return {}

        records.sort(key=lambda x: x["score"])
        n_long = max(1, int(len(records) * LONG_FRACTION))
        n_short = max(1, int(len(records) * SHORT_FRACTION))

        short_side = records[:n_short]
        long_side = records[-n_long:]

        weights = {}

        if PORTFOLIO_METHOD == "equal_weight":
            for r in long_side:
                weights[r["symbol"]] = 0.5 / len(long_side)
            for r in short_side:
                weights[r["symbol"]] = -0.5 / len(short_side)

        else:
            # inverse-volatility / risk-parity proxy
            long_inv = np.array([1.0 / r["vol"] for r in long_side], dtype=float)
            short_inv = np.array([1.0 / r["vol"] for r in short_side], dtype=float)
            long_inv = long_inv / long_inv.sum()
            short_inv = short_inv / short_inv.sum()

            for r, w in zip(long_side, long_inv):
                weights[r["symbol"]] = float(0.5 * w)
            for r, w in zip(short_side, short_inv):
                weights[r["symbol"]] = float(-0.5 * w)

            if PORTFOLIO_METHOD == "volatility_target":
                approx_vol = np.sqrt(
                    sum(
                        (weights[r["symbol"]] * r["vol"]) ** 2
                        for r in long_side + short_side
                    )
                )
                if approx_vol > 0:
                    scale = min(1.0, TARGET_DAILY_VOL / approx_vol)
                    weights = {s: w * scale for s, w in weights.items()}

        return weights

    def _rebalance(self):
        if self.live_mode:
            raise RuntimeError("NB07 governance denial: live execution is prohibited.")
        if self.is_warming_up or self._champion_model is None:
            return

        latest = self._latest_features()
        if latest.empty:
            return

        proposed = self._portfolio_weights(latest)
        approved = self._risk.review(self, proposed)

        for symbol in self._symbols:
            self.set_holdings(symbol, float(approved.get(symbol, 0.0)))

        self._record(
            "REBALANCE",
            {
                "champion": self._champion_name,
                "proposed_positions": len(proposed),
                "approved_positions": len(approved),
                "risk_report": self._risk.last_report,
            },
        )

    def on_dividends(self, dividends):
        for symbol, dividend in dividends.items():
            if symbol in self._event_dates:
                self._event_dates[symbol]["dividend"] = self.time.date()
                self._record(
                    "DIVIDEND",
                    {
                        "symbol": str(symbol),
                        "distribution": float(dividend.distribution),
                    },
                )

    def on_splits(self, splits):
        for symbol, split in splits.items():
            if symbol in self._event_dates:
                self._event_dates[symbol]["split"] = self.time.date()
                self._record(
                    "SPLIT",
                    {
                        "symbol": str(symbol),
                        "split_factor": float(split.split_factor),
                        "type": str(split.type),
                    },
                )

    def on_order_event(self, order_event):
        if order_event.status in [
            OrderStatus.FILLED,
            OrderStatus.PARTIALLY_FILLED,
            OrderStatus.CANCELED,
            OrderStatus.INVALID,
        ]:
            self._record(
                "ORDER_EVENT",
                {
                    "order_id": order_event.order_id,
                    "symbol": str(order_event.symbol),
                    "status": str(order_event.status),
                    "fill_quantity": float(order_event.fill_quantity),
                    "fill_price": float(order_event.fill_price),
                },
            )

    def on_data(self, data):
        # Explicitly keep live denial inside the event loop as a second boundary.
        if self.live_mode:
            raise RuntimeError("NB07 governance denial: live execution is prohibited.")

    def on_end_of_algorithm(self):
        bundle = {
            "notebook": "NB07",
            "bridge": "QuantConnect/LEAN",
            "status": "BACKTEST_RESEARCH_ONLY",
            "live_trading_authorized": False,
            "deployment_authorized": False,
            "champion": self._champion_name,
            "model_diagnostics": self._model_diagnostics,
            "risk_report": self._risk.last_report,
            "audit_events": self._audit,
            "audit_head": stable_hash(self._audit),
            "configuration_hash": stable_hash({
                "models": MODEL_FAMILIES,
                "features": FEATURES,
                "portfolio_method": PORTFOLIO_METHOD,
                "universe": UNIVERSE_BY_SECTOR,
            }),
        }

        try:
            self.object_store.save(
                AUDIT_OBJECT_STORE_KEY,
                json.dumps(bundle, sort_keys=True, default=str),
            )
        except Exception as exc:
            self.log(f"NB07 Object Store warning: {exc}")

        self.log(
            "NB07 COMPLETE: research-only QuantConnect backtest; "
            f"champion={self._champion_name}; live authority=DENIED"
        )
"""

# The generator replaces the embedded defaults with the compiled specification.
def render_strategy_config(spec):
    groups = json.dumps(spec["universe_groups"], indent=4)
    features = json.dumps(spec["feature_contract"], indent=4)
    models = json.dumps(spec["model_families"], indent=4)

    return f"""# Auto-generated by NB07. Research/backtest only.

SEED = {spec['seed']}
RESEARCH_ONLY = True
DEPLOYMENT_AUTHORIZED = False
LIVE_TRADING_AUTHORIZED = False

START_DATE = {tuple(map(int, spec['start_date'].split('-')))}
END_DATE = {tuple(map(int, spec['end_date'].split('-')))}
STARTING_CASH = {spec['starting_cash']}
BENCHMARK = {spec['benchmark']!r}

RESOLUTION = {spec['resolution']!r}
DATA_NORMALIZATION = {spec['data_normalization']!r}
WARMUP_BARS = {spec['warmup_bars']}
TRAINING_LOOKBACK_BARS = {spec['training_lookback_bars']}
RETRAIN_FREQUENCY = "MONTHLY"
REBALANCE_FREQUENCY = "MONTHLY"

FEATURES = {features}
MODEL_FAMILIES = {models}
CHAMPION_SELECTION_RULE = {spec['champion_selection_rule']!r}

SIGNAL_MODE = "CROSS_SECTIONAL_LONG_SHORT"
LONG_FRACTION = 0.20
SHORT_FRACTION = 0.20
MIN_SIGNAL_ABS = 0.00

PORTFOLIO_METHOD = {spec['portfolio_method']!r}
TARGET_DAILY_VOL = 0.010
MAX_GROSS = {spec['risk_limits']['max_gross']}
MAX_POSITION_WEIGHT = {spec['risk_limits']['max_position_weight']}
MAX_SECTOR_GROSS = {spec['risk_limits']['max_sector_gross']}
MAX_PORTFOLIO_DRAWDOWN = {spec['risk_limits']['max_portfolio_drawdown']}
EVENT_COOLDOWN_DAYS = 2

PER_ORDER_FEE_USD = {spec['cost_model']['fee_usd_per_order']}
VOLUME_SHARE_LIMIT = {spec['slippage_model']['volume_limit']}
VOLUME_SHARE_PRICE_IMPACT = {spec['slippage_model']['price_impact']}

UNIVERSE_BY_SECTOR = {groups}

AUDIT_OBJECT_STORE_KEY = "NB07/audit_bundle.json"
"""

strategy_config_path = NB07_ROOT / "strategy_config.py"
main_path = NB07_ROOT / "main.py"
spec_path = NB07_ROOT / "strategy_spec.json"

strategy_config_path.write_text(render_strategy_config(STRATEGY_SPEC))
main_path.write_text(MAIN_SOURCE)
spec_path.write_text(json.dumps(STRATEGY_SPEC, indent=2, sort_keys=True))

print("Generated:", strategy_config_path)
print("Generated:", main_path)
print("Generated:", spec_path)

Generated: /content/nb07_quantconnect_bridge/strategy_config.py
Generated: /content/nb07_quantconnect_bridge/main.py
Generated: /content/nb07_quantconnect_bridge/strategy_spec.json


## 11. Semantic reconciliation: notebook research vs. LEAN execution

A bridge is trustworthy only if it makes differences visible. NB07 therefore maintains a reconciliation contract. Some quantities should match closely—universe membership, feature definitions, model family, rebalance schedule, position bounds, risk thresholds. Other quantities are expected to differ because LEAN models orders, fills, calendars, corporate actions, and reality models differently from a vectorized notebook.

The reconciliation report below does **not** claim that a QuantConnect backtest has already run. It states what must be checked after the platform run.

In [ ]:
RECONCILIATION_CONTRACT = {
    "must_match": {
        "universe_size": 30,
        "feature_contract": STRATEGY_SPEC["feature_contract"],
        "model_families": STRATEGY_SPEC["model_families"],
        "portfolio_method": STRATEGY_SPEC["portfolio_method"],
        "max_gross": STRATEGY_SPEC["risk_limits"]["max_gross"],
        "max_position_weight": STRATEGY_SPEC["risk_limits"]["max_position_weight"],
        "live_trading_authorized": False,
    },
    "compare_with_tolerance_after_platform_run": {
        "signal_dates": "same rebalance schedule after warm-up",
        "security_eligibility": "same event-cooldown rule",
        "turnover": "platform fills may differ from vectorized approximation",
        "transaction_costs": "LEAN reality models govern platform values",
        "equity_curve": "must be reconciled, not assumed identical",
        "drawdown": "must reflect LEAN portfolio accounting",
    },
    "platform_only_evidence": [
        "QuantConnect compilation result",
        "LEAN runtime logs",
        "order tickets and fills",
        "backtest statistics",
        "Object Store audit artifact",
    ],
}
print(json.dumps(RECONCILIATION_CONTRACT, indent=2))

{
  "must_match": {
    "universe_size": 30,
    "feature_contract": [
      "ret_1",
      "mom_5",
      "mom_20",
      "vol_20",
      "vol_60",
      "volume_z20",
      "dollar_volume_z20",
      "ma_ratio_20",
      "ma_ratio_50"
    ],
    "model_families": [
      "logistic_regression",
      "knn",
      "random_forest",
      "mlp",
      "linear_regression"
    ],
    "portfolio_method": "inverse_volatility",
    "max_gross": 1.0,
    "max_position_weight": 0.1,
    "live_trading_authorized": false
  },
  "compare_with_tolerance_after_platform_run": {
    "signal_dates": "same rebalance schedule after warm-up",
    "security_eligibility": "same event-cooldown rule",
    "turnover": "platform fills may differ from vectorized approximation",
    "transaction_costs": "LEAN reality models govern platform values",
    "equity_curve": "must be reconciled, not assumed identical",
    "drawdown": "must reflect LEAN portfolio accounting"
  },
  "platform_only_evidence": [
    "Quant

## 12. Falsification matrix for the bridge

The rewritten NB07 treats code generation itself as a hypothesis that can fail. The bridge is acceptable only if the generated project survives tests designed to reject it. These tests do not prove profitability. They test whether the translation preserves authority, semantics, reproducibility, and the minimum implementation controls.

The matrix includes the failure modes that mattered throughout the course: semantic incompleteness, unregistered capability, leakage, disappearing costs, broken risk independence, live-mode bypass, corporate-action omission, non-reproducible generation, unauthorized network access, and self-promotion.

In [ ]:
FORBIDDEN_CODE_PATTERNS = [
    "requests.",
    "urllib.",
    "socket.",
    "websocket",
    "OPENAI_API_KEY",
    "brokerage",
]

def validate_generated_project(root):
    report = {}

    main_source = (root / "main.py").read_text()
    config_source = (root / "strategy_config.py").read_text()
    spec = json.loads((root / "strategy_spec.json").read_text())

    # Syntax is platform-independent and can be checked locally.
    ast.parse(main_source)
    ast.parse(config_source)
    report["python_syntax"] = True

    report["strategy_spec_complete"] = all(
        field in spec for field in REQUIRED_STRATEGY_FIELDS
    )
    report["universe_size_30"] = len(spec["universe"]) == 30
    report["unique_universe"] = len(set(spec["universe"])) == 30
    report["models_allowlisted"] = set(spec["model_families"]).issubset(APPROVED_MODELS)
    report["features_allowlisted"] = set(spec["feature_contract"]).issubset(APPROVED_FEATURES)
    report["portfolio_allowlisted"] = spec["portfolio_method"] in APPROVED_PORTFOLIOS
    report["human_promotion_only"] = spec["promotion_view"] == "HUMAN_REVIEW_REQUIRED"
    report["live_authority_denied"] = (
        spec["live_trading_authorized"] is False
        and "if self.live_mode" in main_source
    )
    report["deployment_denied"] = spec["deployment_authorized"] is False
    report["independent_risk_present"] = "class IndependentRiskGovernor" in main_source
    report["champion_is_deterministic"] = "balanced_accuracy_score" in main_source
    report["corporate_actions_present"] = (
        "def on_dividends" in main_source
        and "def on_splits" in main_source
    )
    report["cost_models_present"] = (
        "ConstantFeeModel" in main_source
        and "VolumeShareSlippageModel" in main_source
    )
    report["audit_present"] = (
        "def _record" in main_source
        and "on_end_of_algorithm" in main_source
        and "object_store.save" in main_source
    )
    report["no_llm_inside_lean_runtime"] = (
        "from openai" not in main_source.lower()
        and "OpenAI(" not in main_source
    )
    report["no_external_network_code"] = not any(
        pattern.lower() in main_source.lower()
        for pattern in FORBIDDEN_CODE_PATTERNS
    )

    first_hash = stable_hash({
        "main": main_source,
        "config": config_source,
        "spec": spec,
    })
    second_hash = stable_hash({
        "main": (root / "main.py").read_text(),
        "config": (root / "strategy_config.py").read_text(),
        "spec": json.loads((root / "strategy_spec.json").read_text()),
    })
    report["reproducible_hash"] = first_hash == second_hash
    report["artifact_hash"] = first_hash

    report["passed"] = all(
        value is True
        for key, value in report.items()
        if key not in {"artifact_hash", "passed"}
    )
    return report

VALIDATION_REPORT = validate_generated_project(NB07_ROOT)
print(json.dumps(VALIDATION_REPORT, indent=2))
assert VALIDATION_REPORT["passed"]

{
  "python_syntax": true,
  "strategy_spec_complete": true,
  "universe_size_30": true,
  "unique_universe": true,
  "models_allowlisted": true,
  "features_allowlisted": true,
  "portfolio_allowlisted": true,
  "human_promotion_only": true,
  "live_authority_denied": true,
  "deployment_denied": true,
  "independent_risk_present": true,
  "champion_is_deterministic": true,
  "corporate_actions_present": true,
  "cost_models_present": true,
  "audit_present": true,
  "no_llm_inside_lean_runtime": true,
  "no_external_network_code": true,
  "reproducible_hash": true,
  "artifact_hash": "691690ab4e288022df3d29870ce31490bd9151db04323d05533ace82babd9ba5",
  "passed": true
}


## 13. Experiment registry and audit package

The experiment registry prevents a successful backtest from overwriting failed or null experiments. Each experiment has an identity, specification hash, code hash, status, evidence boundary, and promotion state. QuantConnect results are added only after an actual platform run. Until then, the registry says `READY_FOR_QUANTCONNECT_BACKTEST`, not `PASS`.

In [ ]:
EXPERIMENT = {
    "experiment_id": "NB07-QC-EXP-001",
    "created_at": datetime.utcnow().isoformat() + "Z",
    "objective": MISSION["objective"],
    "strategy_spec_hash": STRATEGY_SPEC_HASH,
    "generated_project_hash": VALIDATION_REPORT["artifact_hash"],
    "status": "READY_FOR_QUANTCONNECT_BACKTEST",
    "platform_compile_status": "NOT_RUN",
    "platform_backtest_status": "NOT_RUN",
    "research_only": True,
    "deployment_authorized": False,
    "live_trading_authorized": False,
    "promotion_view": "HUMAN_REVIEW_REQUIRED",
    "required_post_run_evidence": RECONCILIATION_CONTRACT["platform_only_evidence"],
}

EXPERIMENT_REGISTRY = {
    "registry_version": "2.0",
    "experiments": [EXPERIMENT],
    "registry_hash": None,
}
EXPERIMENT_REGISTRY["registry_hash"] = stable_hash(EXPERIMENT_REGISTRY["experiments"])

AUDIT_BUNDLE = {
    "notebook": "NB07",
    "title": "Governed QuantConnect / LEAN Implementation Bridge",
    "mission_gate": MISSION_GATE,
    "plan_proposal_hash": stable_hash(PLAN_PROPOSAL),
    "strategy_spec_hash": STRATEGY_SPEC_HASH,
    "validation_report": VALIDATION_REPORT,
    "reconciliation_contract": RECONCILIATION_CONTRACT,
    "experiment_registry_hash": EXPERIMENT_REGISTRY["registry_hash"],
    "authority": {
        "research_only": True,
        "deployment_authorized": False,
        "live_trading_authorized": False,
        "promotion_view": "HUMAN_REVIEW_REQUIRED",
    },
}

(NB07_ROOT / "experiment_registry.json").write_text(
    json.dumps(EXPERIMENT_REGISTRY, indent=2, sort_keys=True)
)
(NB07_ROOT / "validation_report.json").write_text(
    json.dumps(VALIDATION_REPORT, indent=2, sort_keys=True)
)
(NB07_ROOT / "reconciliation_contract.json").write_text(
    json.dumps(RECONCILIATION_CONTRACT, indent=2, sort_keys=True)
)
(NB07_ROOT / "audit_bundle.json").write_text(
    json.dumps(AUDIT_BUNDLE, indent=2, sort_keys=True)
)

print("Experiment status:", EXPERIMENT["status"])
print("Promotion:", EXPERIMENT["promotion_view"])

Experiment status: READY_FOR_QUANTCONNECT_BACKTEST
Promotion: HUMAN_REVIEW_REQUIRED


/tmp/ipykernel_6896/2587617058.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat() + "Z",


## 14. Optional QuantConnect Research probe

If this notebook is running inside the QuantConnect Research environment, the following cell can instantiate `QuantBook`, subscribe to a small probe set, and verify that native history access is available. It deliberately does not run a backtest and does not place orders.

In Colab, the cell reports that QuantConnect Research is unavailable and continues normally.

In [ ]:
if QUANTCONNECT_RESEARCH_AVAILABLE:
    qb = QuantBook()
    probe = [
        qb.add_equity("SPY", Resolution.DAILY).symbol,
        qb.add_equity("AAPL", Resolution.DAILY).symbol,
    ]
    probe_history = qb.history(probe, 10, Resolution.DAILY)
    print("QuantConnect Research probe rows:", len(probe_history))
    display(probe_history.tail())
else:
    print("QuantConnect Research runtime not detected. Project generation remains fully available.")

QuantConnect Research runtime not detected. Project generation remains fully available.


## 15. Export the QuantConnect project

The package contains the full notebook plus the compact project files that should be copied into a QuantConnect Python project. The generated algorithm is intentionally **backtest-only**. A platform compile or backtest result must be added to the registry after it actually exists; NB07 never fabricates platform evidence.

In [ ]:
README_TEXT = r'''# NB07 — QuantConnect / LEAN Governed Implementation Bridge

This package is generated by the rewritten NB07 notebook.

## Purpose

Translate a governed NB10-like research specification into a QuantConnect/LEAN Python backtest project while preserving:

- deterministic validation and compilation;
- heterogeneous model comparison;
- mathematical champion selection;
- point-in-time trailing features;
- corporate-action awareness;
- explicit fee and slippage modeling;
- independent risk logic;
- audit/provenance artifacts;
- research-only authority.

## QuantConnect project files

- `main.py` — LEAN algorithm.
- `strategy_config.py` — deterministic compiled configuration.
- `strategy_spec.json` — machine-readable research/implementation contract.
- `experiment_registry.json` — immutable experiment metadata.
- `validation_report.json` — compiler/falsification checks.
- `research.ipynb` — the full pedagogical bridge notebook.

## How to use

1. Create a Python project in QuantConnect Cloud.
2. Replace the project `main.py` with this package's `main.py`.
3. Add `strategy_config.py` to the same project.
4. Optionally upload `research.ipynb` as the project's research notebook.
5. Run a **backtest**.
6. Review the LEAN result together with `strategy_spec.json`, risk limits, model diagnostics, corporate-action logs, and the Object Store audit artifact.
7. Do not deploy live from this package. The generated algorithm intentionally fails if `live_mode` is true.

## Important semantic limits

The default 30-security universe is a stable teaching fixture, not a point-in-time institutional universe and not an investment recommendation. Production research should replace it with a governed universe-selection process.

The notebook can validate Python syntax and institutional invariants locally. It cannot certify QuantConnect cloud compilation or matched-data backtest results without actually running the project on QuantConnect.

## Current QuantConnect API concepts used

The generated project follows current QuantConnect documentation for:

- `QCAlgorithm.initialize`;
- `add_equity` and historical `history` requests;
- Scheduled Events;
- `train` for ML training;
- fee and slippage reality models;
- dividends and splits event handlers;
- Object Store persistence.

Research/backtest only. Live trading and deployment are unauthorized.
'''
(NB07_ROOT / "README_QUANTCONNECT.md").write_text(README_TEXT)

print("Files ready for export:")
for path in sorted(NB07_ROOT.iterdir()):
    if path.is_file():
        print(" -", path.name, path.stat().st_size, "bytes")

Files ready for export:
 - README_QUANTCONNECT.md 2373 bytes
 - audit_bundle.json 2634 bytes
 - experiment_registry.json 1358 bytes
 - main.py 20432 bytes
 - reconciliation_contract.json 1151 bytes
 - strategy_config.py 1864 bytes
 - strategy_spec.json 3643 bytes
 - validation_report.json 679 bytes


# Conclusion — from autonomous research to governed platform implementation

NB07 completes a conceptual movement that began much earlier in the course. At first, the problem looked like model building: create data, calculate features, fit an estimator, evaluate predictions. The architecture then widened. A strategy required portfolio construction. A portfolio required execution assumptions. Execution required independent risk. Reproducibility required provenance. Multiple capabilities required registries, tools, skills, and agents. Substantial objectives required constellations and a meta-agent. Persistent autonomy required state, budgets, recovery, checkpoints, and audit. The later integrated notebooks added a real LLM, but only inside a bounded cognitive role. NB10 finally showed the complete research institution. NB07 now asks whether that institution can express itself inside a real algorithmic platform without losing the distinctions that made the architecture trustworthy.

The answer is not “generate trading code.” Code generation is the least interesting part of the problem. The difficult task is preserving **semantic identity** across computational environments. A return calculated in a DataFrame and a return implied by an event-driven sequence are not automatically the same object. A portfolio weight in a research table and a filled position in LEAN are not automatically the same state. A corporate action represented as a database record and a corporate action delivered by the platform event loop have different operational consequences. A cost assumption in a spreadsheet and a slippage model attached to a `Security` object belong to different layers. NB07 therefore treats translation as an engineering discipline.

The expanded `StrategySpec` is the central artifact of that discipline. It makes the research hypothesis inspectable before platform execution begins. It states the universe, resolution, data normalization, features, labels, model families, champion-selection rule, training schedule, signal transformation, portfolio method, cost assumptions, risk limits, corporate-action policy, expected outputs, limitations, and authority boundary. By carrying these semantics explicitly, the specification prevents the code generator from quietly choosing material assumptions. A missing material field is a failed compilation, not an invitation to guess.

This explicitness also improves the role of the LLM. A language model is useful when the human objective is broad: it can interpret the research question, identify relevant approved model families, propose experimental branches, and articulate tests. But NB07 never lets that proposal become a direct trading instruction. The proposal is normalized and validated. Unknown capabilities fail. Prohibited authority fails. The deterministic compiler chooses the implementable representation. The generated LEAN project contains no OpenAI client and no mechanism for asking an LLM what order to place. This separation preserves the principle established in NB09 and NB10: **LLM intelligence belongs in the control plane; deterministic execution belongs in the execution plane.**

The QuantConnect version also clarifies what “realism” means. Realism does not mean merely replacing synthetic prices with historical prices. It means inheriting an event-driven calendar, market subscriptions, corporate actions, order events, portfolio accounting, fee models, slippage models, and platform resource constraints. The implementation is therefore closer to the operational environment in which a strategy would actually be evaluated. At the same time, the notebook refuses to confuse this additional realism with evidence of profitability. A more realistic backtest can reject a strategy more credibly; it cannot guarantee future returns.

The model laboratory is intentionally plural. Logistic regression, KNN, random forest, MLP, and linear regression remain candidates because the course is not teaching students to worship a single fashionable algorithm. They encode different inductive biases. The training function evaluates them on a chronological validation segment built entirely from trailing history. The selector uses a predefined quantitative metric and tie-breaker. This matters because the system must be able to say that a simple model won, that a complex model failed, or that all candidates were weak. A governed research institution has to preserve null results.

The distinction between model, signal, portfolio, and risk remains equally important on QuantConnect. A model score does not become an order merely because it is positive. Scores are transformed into a cross-sectional strategy. The strategy becomes a candidate portfolio under a declared weighting method. The independent risk governor then reviews the candidate portfolio. Only approved weights reach `set_holdings`. This decomposition makes failure diagnosis possible. If the backtest is poor, we can ask whether prediction failed, the cross-sectional transformation failed, the portfolio method amplified noise, costs consumed the edge, corporate events created instability, or the risk limits altered exposure. Without this decomposition, “the strategy failed” is too crude to be useful.

Corporate actions provide a particularly good example of why the bridge matters. In NB00 they were explicit rows in a controlled synthetic database. In the integrated research system they became event-aware information subject to point-in-time discipline. In LEAN they are delivered through platform event handlers and interact with normalized prices and portfolio state. NB07 records them and temporarily excludes affected securities from new allocations. The precise policy is less important than the architectural lesson: event handling must be explicit, temporally valid, and testable.

The same is true of costs. The notebook does not hide transaction costs inside a single scalar deduction at the end of the experiment. The generated platform project assigns fee and slippage models to each security. These models can be changed through the specification and then falsified through sensitivity tests. If the strategy survives only when costs are zero, that is useful evidence against the strategy. The objective of the research institution is not to rescue an idea; it is to discover where it breaks.

The falsification matrix therefore becomes a bridge-level acceptance mechanism. Syntax must parse. The strategy specification must be complete. All models and features must come from allowlists. Live authority must remain false. The independent risk class must exist. Corporate-action handlers must exist. Cost models must exist. Audit logic must exist. No LLM may be imported into the LEAN runtime. No external network library may silently appear. Re-running the compiler with the same inputs must produce the same artifact hash. These tests do not prove that QuantConnect will accept every semantic choice forever; APIs evolve and the platform remains the ultimate compilation authority. What they prove is that the notebook has translated its declared architecture consistently.

The experiment registry is equally important. Platform execution creates a temptation to keep only the attractive run. NB07 resists that temptation by defining the experiment before the platform result exists. Its initial state is `READY_FOR_QUANTCONNECT_BACKTEST`. Compilation and backtest statuses are `NOT_RUN`. After the platform run, those fields can be updated with actual evidence. A failed compile, a weak backtest, a risk halt, or a null result remains part of the institutional record. This is how an autonomous research system avoids becoming an autonomous confirmation-bias machine.

There is also a deeper lesson about autonomy. Crossing into QuantConnect increases capability dramatically. The system can now materialize a project in the vocabulary of a mature event-driven engine. It can train models under platform time, process corporate actions, create holdings, model costs, and record order events. Yet its authority has not increased. Deployment remains false. Live trading remains false. Promotion remains `HUMAN_REVIEW_REQUIRED`. This asymmetry—more capability without automatic expansion of authority—is precisely what makes the architecture credible.

For students of autonomous systems, NB07 therefore demonstrates that implementation is not the abandonment of governance. It is where governance becomes most valuable. A bounded system should be able to cross interfaces, translate representations, and execute complex workflows while preserving the original mandate. The relevant question is not whether the machine can place an order. The relevant question is whether every material transition from idea to data, from data to feature, from feature to prediction, from prediction to position, and from position to simulated order remains attributable, constrained, reproducible, falsifiable, and interruptible.

For quantitative finance, the notebook also exposes why productionization is its own research problem. The closer we move to a platform, the more hidden assumptions become visible: scheduling, stale data, warm-up, security identifiers, normalization, corporate actions, training windows, cash, leverage, fees, slippage, order sequencing, portfolio state, resource budgets, and persistence. A strategy that exists only as a vectorized notebook function has not yet survived these semantics. NB07 makes those semantics part of the test.

The appropriate next step after running this notebook is not live deployment. It is a **QuantConnect cloud backtest followed by reconciliation**. The generated `main.py` and `strategy_config.py` should be copied into a Python project. The platform should compile the project. The resulting logs, fills, statistics, model diagnostics, corporate-action events, and Object Store artifact should then be compared with the `StrategySpec` and reconciliation contract. Discrepancies should be explained, not averaged away. Only after the bridge behaves as intended should the research hypothesis itself be judged.

Even a successful backtest remains only one piece of evidence. Robustness should be tested across alternative periods, costs, universe definitions, model subsets, portfolio methods, event policies, and risk limits. The falsification matrix should grow, not shrink, as the system approaches a material decision. Independent review should remain independent. Security and operational controls should become stronger. A legal and institutional authority chain would be required before any production use.

The final result of NB07 is therefore not a trading robot. It is something more disciplined: a **governed research-to-platform compiler**. It connects human purpose and machine intelligence to a real algorithmic execution environment while preserving the boundary between research and authority. It can produce a backtest-ready QuantConnect project. It cannot promote itself. It cannot grant itself deployment permission. It cannot convert a promising equity curve into a mandate.

That is the correct ending for this stage of the architecture. We have moved from a market database to models, from models to strategies, from strategies to agents, from agents to an autonomous research institution, from an autonomous institution to an LLM-governed control plane, and now from that control plane to a platform-executable research artifact. The system is more capable because it can cross the boundary. It is trustworthy only because the boundary remains visible.

> **Final NB07 doctrine:** compile meaning before compiling code; preserve point-in-time evidence; separate cognition from execution; select champions mathematically; keep risk independent; make costs and corporate actions explicit; register every experiment; fail closed on authority; and require a human decision before any move from research into production.

**QuantConnect backtesting is the next experiment. It is not deployment authorization.**

# Appendix A: The QuantConnect Trade Logic


## QuantConnect Trading Logic — `main.py` + `strategy_config.py`

## Overview

The QuantConnect implementation is a **monthly retrained, cross-sectional, market-neutral U.S. equity strategy**.

The algorithm does not rely on one fixed model. Instead, it repeatedly trains several approved machine-learning models, evaluates them on a chronological validation sample, selects the strongest current model as the **champion**, uses that champion to rank the stock universe, builds long and short baskets from the strongest and weakest names, sizes those positions using a risk-aware portfolio rule, and then sends the proposed portfolio through an independent risk governor before QuantConnect/LEAN is allowed to execute any holdings changes.

The entire trading logic can be summarized as:

**Historical market data → engineered features → competing models → champion selection → stock ranking → long/short portfolio → independent risk review → LEAN execution**

---

# 1. Investment Universe

The algorithm trades a fixed universe of **30 U.S. equities**, divided into six groups of five securities each.

## Technology

- AAPL
- MSFT
- NVDA
- AVGO
- ORCL

## Financials

- JPM
- BAC
- GS
- MS
- C

## Healthcare

- JNJ
- UNH
- LLY
- MRK
- ABBV

## Industrials

- CAT
- GE
- HON
- UPS
- RTX

## Consumer

- AMZN
- WMT
- COST
- MCD
- HD

## Energy

- XOM
- CVX
- COP
- SLB
- EOG

Therefore:

**6 sectors × 5 securities = 30 stocks**

The strategy is fundamentally **cross-sectional**.

It does not ask only:

> Will one particular stock go up?

Instead, it asks:

> Which stocks look relatively strongest and which look relatively weakest within the 30-stock universe?

The portfolio is then constructed from those relative rankings.

---

# 2. Data Frequency

The system operates on **daily data**.

For each security, QuantConnect provides historical observations including:

- adjusted close price,
- volume,
- dividend events,
- split events,
- trading calendar information.

The algorithm uses approximately:

- **80 daily bars for warm-up**
- **320 daily bars for model training**

The warm-up ensures that rolling indicators and features have enough observations before the strategy begins making decisions.

The training window represents approximately a little more than one year of daily trading history.

---

# 3. Feature Engineering

The strategy transforms raw price and volume history into a set of predictive variables.

The feature set is:

- `ret_1`
- `mom_5`
- `mom_20`
- `vol_20`
- `vol_60`
- `volume_z20`
- `dollar_volume_z20`
- `ma_ratio_20`
- `ma_ratio_50`

These features can be divided into four conceptual groups.

---

## 3.1 One-Day Return

The feature:

`ret_1`

measures the most recent daily return.

Conceptually:

$
r_{1,t} = \frac{P_t}{P_{t-1}} - 1
$

It tells the model whether the security has just moved upward or downward.

---

## 3.2 Momentum

The strategy calculates:

`mom_5`

and:

`mom_20`

These measure cumulative price changes over approximately one trading week and one trading month.

Conceptually:

$
mom_{5,t} = \frac{P_t}{P_{t-5}} - 1
$

$
mom_{20,t} = \frac{P_t}{P_{t-20}} - 1
$

These variables attempt to capture short- and medium-term persistence in price movements.

---

## 3.3 Volatility

The system calculates:

`vol_20`

and:

`vol_60`

These are rolling standard deviations of daily returns over approximately one month and three months.

They tell the system whether a stock is currently experiencing relatively calm or turbulent behavior.

Volatility is used both as predictive information and later during portfolio construction.

---

## 3.4 Volume Abnormality

The strategy calculates:

`volume_z20`

and:

`dollar_volume_z20`

The first compares current trading volume with its recent 20-day distribution.

The second performs a similar calculation using dollar trading volume.

A positive z-score indicates unusually high trading activity.

A negative z-score indicates below-normal activity.

These variables attempt to identify unusual market participation.

---

## 3.5 Moving-Average Position

The features:

`ma_ratio_20`

and:

`ma_ratio_50`

compare the current price with its 20-day and 50-day moving averages.

Conceptually:

$
ma\_ratio_{20} = \frac{P_t}{MA_{20,t}} - 1
$

$
ma\_ratio_{50} = \frac{P_t}{MA_{50,t}} - 1
$

A positive value means the stock trades above its moving average.

A negative value means it trades below it.

These features capture trend location in addition to simple cumulative momentum.

---

# 4. Prediction Target

The model attempts to forecast the stock's **next-day return**.

The regression label is approximately:

$
r_{t+1}
$

The classification label is:

$
y_{t+1} =
\begin{cases}
1 & \text{if } r_{t+1} > 0 \\
0 & \text{otherwise}
\end{cases}
$

Therefore, each observation asks:

> Given the information available today, can the model infer whether tomorrow's return is likely to be positive?

For classifiers, the output is a directional probability or class prediction.

For the regression model, the output is a predicted future return.

---

# 5. Candidate Model Library

The strategy trains five competing model families:

1. Logistic Regression
2. K-Nearest Neighbors
3. Random Forest
4. Multi-Layer Perceptron
5. Linear Regression

The models all use the same underlying feature space.

This is important because the system is not testing different data sets for each model.

It is testing whether different mathematical structures can extract different predictive information from the same market evidence.

---

## 5.1 Logistic Regression

Logistic regression estimates the probability that the next-day return is positive.

It is a relatively simple linear probabilistic classifier.

Its strengths include:

- transparency,
- stability,
- low complexity,
- strong baseline behavior.

---

## 5.2 K-Nearest Neighbors

KNN searches historical observations that resemble the current feature vector.

The intuition is:

> When market conditions looked similar in the past, what happened next?

It is non-parametric and can capture local nonlinear behavior.

---

## 5.3 Random Forest

Random Forest combines many decision trees.

It can capture:

- nonlinear relationships,
- interactions between variables,
- threshold effects,
- regime-dependent behavior.

It is often particularly useful when predictive relationships are not approximately linear.

---

## 5.4 Multi-Layer Perceptron

The MLP is a neural-network model.

It can represent complex nonlinear relationships between momentum, volatility, volume, and trend variables.

Because neural networks can overfit relatively easily in financial data, it competes against simpler models rather than automatically receiving preference.

---

## 5.5 Linear Regression

Linear regression predicts the magnitude of the next-day return.

Its predicted value is converted into a directional signal by looking at whether the forecast is positive or negative.

This gives the model competition both classification and regression approaches.

---

# 6. Chronological Training and Validation

The dataset is sorted chronologically.

Approximately:

**first 80% → model fitting**

**last 20% → validation**

This is deliberate.

The algorithm does not randomly shuffle financial observations.

The chronological split attempts to reproduce the actual forecasting question:

> Could a model trained on earlier market information have predicted later observations?

This reduces the danger of introducing time leakage through random train/test mixing.

---

# 7. Champion Selection

Each model is evaluated on the chronological validation sample.

The principal metric is:

`balanced_accuracy_score`

Balanced accuracy gives approximately equal importance to correctly predicting positive and negative classes.

Each candidate model therefore receives a validation score.

For example, a hypothetical training cycle might produce:

| Model | Validation Balanced Accuracy |
|---|---:|
| Logistic Regression | 0.523 |
| KNN | 0.541 |
| Random Forest | 0.566 |
| MLP | 0.548 |
| Linear Regression | 0.517 |

In this example:

**Random Forest becomes the champion.**

The champion is selected mathematically.

There is no LLM deciding which model "sounds better."

The highest validation score wins.

If two models obtain the same score, a predefined complexity ranking is used as the deterministic tie-breaker.

The selected model is stored as:

`self._champion_model`

The champion remains active until the next model-training cycle.

---

# 8. Monthly Retraining

The strategy retrains the model library approximately once per month.

At the beginning of each month:

1. retrieve trailing market history,
2. rebuild all features,
3. construct the training dataset,
4. fit all approved models,
5. evaluate them on the chronological validation set,
6. rank the models,
7. select the new champion.

The champion can therefore change through time.

For example:

January → Random Forest

February → Logistic Regression

March → KNN

April → Random Forest

The system is adaptive in the sense that model choice is periodically re-evaluated rather than permanently fixed.

---

# 9. Generating Stock Scores

At portfolio-rebalance time, the algorithm constructs the latest feature vector for each eligible security.

The current champion then scores every stock.

For classification models, the probability can be transformed approximately as:

$
score = P(\text{up}) - 0.5
$

For example:

$
P(\text{up}) = 0.70
$

becomes:

$
score = +0.20
$

while:

$
P(\text{up}) = 0.35
$

becomes:

$
score = -0.15
$

For Linear Regression, the predicted next-day return itself becomes the score.

The system therefore generates one comparable score per security.

---

# 10. Cross-Sectional Ranking

The stocks are sorted by champion-model score.

The configuration specifies:

`LONG_FRACTION = 0.20`

`SHORT_FRACTION = 0.20`

With 30 securities:

$
30 \times 20\% = 6
$

Therefore, approximately:

- top 6 stocks → long candidates
- middle 18 stocks → flat
- bottom 6 stocks → short candidates

Conceptually:

| Rank | Action |
|---|---|
| 1–6 | Long |
| 7–24 | Flat |
| 25–30 | Short |

The economic idea is not simply to forecast the whole market.

It is to identify relative winners and relative losers.

---

# 11. Long/Short Structure

The strategy is designed to be approximately market neutral.

Roughly:

**+50% gross exposure to long positions**

and:

**−50% gross exposure to short positions**

Therefore:

$
Gross \approx 100\%
$

while:

$
Net \approx 0\%
$

The strategy therefore attempts to profit from the relative performance difference between the long and short baskets rather than from broad equity-market direction.

The core economic bet is:

> the stocks identified as strongest by the champion should outperform the stocks identified as weakest.

---

# 12. Corporate-Action Filter

The algorithm explicitly listens for:

`on_dividends()`

and:

`on_splits()`

When a dividend or split occurs, the event date is recorded.

The configuration contains:

`EVENT_COOLDOWN_DAYS = 2`

A security that has experienced a recent corporate action can therefore be temporarily removed from eligibility for a new position.

The sequence is:

**Corporate event → record event → two-day cool-down → reconsider eligibility**

The model does not receive advance knowledge of future dividends or splits.

The events are handled only when they become known to the LEAN event system.

---

# 13. Portfolio Construction

Stock ranking determines **which securities** belong in the portfolio.

A separate portfolio-construction rule determines **how much capital** each selected security receives.

The default method is:

`PORTFOLIO_METHOD = "inverse_volatility"`

The basic idea is:

$
w_i \propto \frac{1}{\sigma_i}
$

Lower-volatility stocks therefore receive relatively larger weights.

Higher-volatility stocks receive relatively smaller weights.

The long and short sides are normalized separately.

This helps prevent one very volatile security from dominating portfolio risk.

---

# 14. Independent Risk Governor

The raw portfolio is not sent directly to QuantConnect.

It first passes through:

`IndependentRiskGovernor`

This risk component is logically independent from the machine-learning models.

It does not:

- fit models,
- choose the champion,
- generate predictive scores.

It only evaluates whether the proposed portfolio respects institutional risk limits.

---

# 15. Maximum Position Weight

The configuration specifies:

`MAX_POSITION_WEIGHT = 0.10`

Therefore:

$
|w_i| \leq 10\%
$

No security is permitted to exceed a 10% portfolio allocation in absolute value.

If the portfolio engine requests more, the risk governor clamps the position.

---

# 16. Maximum Sector Exposure

The configuration specifies:

`MAX_SECTOR_GROSS = 0.35`

Therefore the total absolute exposure to any one of the six sector groups cannot exceed approximately:

$
35\%
$

For example, if the Technology basket attempted to reach 50% gross exposure, the risk governor would scale those positions downward.

---

# 17. Maximum Gross Exposure

The configuration contains:

`MAX_GROSS = 1.00`

Therefore:

$
\sum_i |w_i| \leq 100\%
$

If the proposed portfolio exceeds this amount, all weights are proportionally scaled down.

---

# 18. Drawdown Risk Halt

The risk engine tracks the highest observed portfolio value.

Let:

$
V_{\max}
$

be the historical portfolio high-water mark.

The current drawdown is approximately:

$
DD_t = \frac{V_t}{V_{\max}} - 1
$

The configuration specifies:

`MAX_PORTFOLIO_DRAWDOWN = 0.12`

Therefore, if:

$
DD_t \leq -12\%
$

the portfolio enters a risk-halt state.

The risk governor then returns no approved target positions.

Conceptually:

**Portfolio drawdown breaches −12% → risk halt → portfolio exposure removed**

---

# 19. Final Portfolio Approval

The complete path from prediction to execution is therefore:

**Model scores**

→ **Cross-sectional ranking**

→ **Long/short candidate selection**

→ **Inverse-volatility weighting**

→ **Position limit check**

→ **Sector concentration check**

→ **Gross-exposure check**

→ **Drawdown check**

→ **Approved portfolio**

Only after all of these stages does the portfolio reach QuantConnect.

---

# 20. LEAN Order Submission

Once approved weights exist, the strategy calls:

`self.set_holdings(symbol, target_weight)`

for each security.

LEAN then calculates the required transaction quantity to move from the current position to the target position.

For securities that are no longer selected, the target becomes:

`0.0`

which causes the previous position to be reduced or closed.

The strategy therefore performs a full portfolio rebalance rather than simply adding new positions.

---

# 21. Rebalance Frequency

The portfolio is rebalanced approximately **monthly**.

This is an important characteristic of the current code.

The prediction target is one day ahead:

**next-day return**

but the portfolio itself is revised monthly.

Therefore:

**prediction horizon ≠ portfolio rebalance horizon**

This is one of the important assumptions that should later be tested in the falsification matrix.

Possible future experiments could compare:

- 1-day prediction / daily rebalance
- 5-day prediction / weekly rebalance
- 20-day prediction / monthly rebalance

The current implementation deliberately preserves the existing monthly design.

---

# 22. Transaction Costs and Slippage

Each security receives an explicit fee model and slippage model.

The current commission configuration is:

`PER_ORDER_FEE_USD = 0.00`

Therefore explicit commissions are currently zero.

However, slippage is modeled through:

`VolumeShareSlippageModel`

with parameters:

`VOLUME_SHARE_LIMIT = 0.025`

`VOLUME_SHARE_PRICE_IMPACT = 0.10`

The model attempts to impose greater execution cost when the requested trade represents a larger share of available market volume.

This creates a basic implementation-realism layer.

---

# 23. Order Events

The algorithm contains:

`on_order_event()`

which records significant order-state changes.

Examples include:

- FILLED
- PARTIALLY_FILLED
- CANCELED
- INVALID

For relevant order events the system records information such as:

- order identifier,
- security,
- order status,
- fill quantity,
- fill price.

This means the strategy maintains an execution-level audit trail rather than only calculating theoretical returns.

---

# 24. Audit Events

The algorithm records several important internal events.

Examples include:

- `INITIALIZED`
- `TRAIN_SKIPPED`
- `TRAIN_FAILED`
- `CHAMPION_SELECTED`
- `REBALANCE`
- `DIVIDEND`
- `SPLIT`
- `ORDER_EVENT`

Each audit event contains:

- timestamp,
- event type,
- payload,
- hash.

At the end of the backtest, the audit information is packaged into an artifact containing:

- final champion,
- model diagnostics,
- risk report,
- audit events,
- audit-chain hash,
- configuration hash,
- research-only status.

---

# 25. Research-Only Governance

The algorithm contains explicit live-trading denial.

The configuration specifies:

`RESEARCH_ONLY = True`

`DEPLOYMENT_AUTHORIZED = False`

`LIVE_TRADING_AUTHORIZED = False`

The algorithm also checks:

`self.live_mode`

If the project is launched in live mode, the code raises an exception.

The implementation is therefore deliberately designed for:

**research and backtesting only**

and not for automatic live deployment.

---

# 26. Economic Hypothesis

Ignoring the software architecture for a moment, the fundamental investment hypothesis is:

> Recent returns, momentum, volatility, trend position, and trading activity contain useful information about the relative near-term attractiveness of stocks.

A second hypothesis is:

> Different model families may exploit this information differently across market conditions.

A third hypothesis is:

> Instead of permanently committing to one model, periodically selecting the empirically strongest model may improve robustness.

A fourth hypothesis is:

> Relative signals may be more useful when expressed through a market-neutral long/short portfolio than through outright directional exposure.

The complete economic chain is therefore:

**Market information**

→ **Predict relative future behavior**

→ **Rank securities**

→ **Buy relative winners**

→ **Short relative losers**

→ **Control risk**

→ **Test whether the long basket outperforms the short basket after implementation costs**

---

# 27. Concrete Example

Suppose the algorithm reaches the beginning of June.

The system retrieves trailing history and trains all five models.

Assume the validation scores are:

| Model | Balanced Accuracy |
|---|---:|
| Logistic Regression | 0.523 |
| KNN | 0.541 |
| Random Forest | 0.566 |
| MLP | 0.548 |
| Linear Regression | 0.517 |

Random Forest becomes the champion.

The Random Forest then scores the 30 securities.

Assume the strongest six are:

- NVDA
- MSFT
- LLY
- JPM
- CAT
- XOM

The weakest six are:

- C
- BAC
- UPS
- HD
- EOG
- ORCL

The signal engine therefore creates:

## Long basket

- NVDA
- MSFT
- LLY
- JPM
- CAT
- XOM

## Short basket

- C
- BAC
- UPS
- HD
- EOG
- ORCL

The portfolio engine then applies inverse-volatility weighting.

A possible raw portfolio might look approximately like:

| Security | Weight |
|---|---:|
| NVDA | +5% |
| MSFT | +10% |
| LLY | +8% |
| JPM | +9% |
| CAT | +9% |
| XOM | +9% |
| C | −9% |
| BAC | −10% |
| UPS | −8% |
| HD | −7% |
| EOG | −8% |
| ORCL | −8% |

These weights are only illustrative.

The actual weights depend on estimated volatility.

The independent risk governor then asks:

- Is any position above 10%?
- Is any sector above 35% gross exposure?
- Is total gross exposure above 100%?
- Has portfolio drawdown exceeded 12%?

If the portfolio passes all checks, the approved weights are sent to LEAN.

LEAN then generates the actual orders required to move from the current portfolio to the new target portfolio.

---

# 28. What Happens the Following Month

The process starts again.

The models are retrained.

New validation scores are calculated.

The champion may remain Random Forest.

Or another model may become champion.

For example:

**June → Random Forest**

**July → KNN**

**August → Logistic Regression**

The portfolio therefore adapts in two dimensions:

1. the ranking of securities changes;
2. the model generating those rankings can also change.

---

# 29. Full Trading Workflow

The entire QuantConnect trading system can be represented as:

**30-stock universe**

↓

**Daily historical data**

↓

**Feature engineering**

- returns
- momentum
- volatility
- volume abnormality
- moving-average position

↓

**Monthly model training**

↓

**Five competing models**

- Logistic Regression
- KNN
- Random Forest
- MLP
- Linear Regression

↓

**Chronological validation**

↓

**Mathematical champion selection**

↓

**Score all eligible stocks**

↓

**Cross-sectional ranking**

↓

**Top 20% → Long**

**Middle 60% → Flat**

**Bottom 20% → Short**

↓

**Inverse-volatility portfolio weighting**

↓

**Corporate-action exclusions**

↓

**Independent risk governor**

- max 10% per stock
- max 35% gross per sector
- max 100% total gross
- halt after 12% drawdown

↓

**Approved target portfolio**

↓

**LEAN `set_holdings()`**

↓

**Orders and fills**

↓

**Execution accounting**

↓

**Audit trail**

↓

**Monthly retraining and repetition**

---

# 30. What Is Distinctive About the Strategy

The most important feature of the implementation is that it is not simply:

> Train a Random Forest and trade its predictions.

Instead, the system separates several distinct functions:

**Data layer**

→ produces historical observations.

**Feature layer**

→ converts those observations into predictive variables.

**Model layer**

→ compares alternative prediction methods.

**Champion-selection layer**

→ mathematically selects the strongest current model.

**Signal layer**

→ converts model outputs into relative security rankings.

**Portfolio layer**

→ converts rankings into target weights.

**Risk layer**

→ independently constrains those weights.

**Execution layer**

→ converts approved targets into QuantConnect orders and fills.

**Audit layer**

→ records what happened.

This layered architecture makes it possible to determine where a strategy succeeds or fails.

For example, poor performance could arise from:

- weak predictive models,
- unstable champion selection,
- poor ranking logic,
- inappropriate portfolio weighting,
- excessive turnover,
- transaction costs,
- corporate-action effects,
- concentration,
- drawdown constraints,
- regime changes.

The system is therefore designed not merely to produce a backtest, but to make the entire decision path inspectable.

---

# 31. Final Summary

The QuantConnect code implements an adaptive cross-sectional machine-learning strategy.

Every month, it asks:

1. What does recent market history tell us?
2. Which predictive model currently performs best?
3. Which stocks appear strongest?
4. Which stocks appear weakest?
5. How should those positions be weighted?
6. Do those weights survive independent risk constraints?
7. What orders are required to reach the approved portfolio?

The ultimate trading rule is therefore:

**Train several models on trailing data → select the best validated model → rank the 30-stock universe → buy the top 20% → short the bottom 20% → size positions by inverse volatility → enforce concentration and drawdown limits → rebalance through LEAN → repeat monthly.**

The deeper architecture can be summarized as:

**DATA → FEATURES → MODELS → CHAMPION → SIGNALS → PORTFOLIO → RISK → EXECUTION → AUDIT**

That is the trading logic embodied by `main.py` and `strategy_config.py`.


#Appendix B. The role of AI

## How Artificial Intelligence Participated in the Investment Process  
## Explanation for the Investment Committee

The most important point for the Investment Committee is that **artificial intelligence was not used to make an unconstrained discretionary investment decision**. The system was designed so that AI participated principally in **research, model comparison, hypothesis generation, strategy construction, and critical evaluation**, while the actual selection of the investment model and portfolio was governed by deterministic quantitative rules, risk limits, validation tests, and ultimately human authority.

A useful way to understand the architecture is:

> **AI helped us search the space of possible investment strategies. Mathematics and governance determined which alternatives survived.**

This distinction is central.

---

# 1. We Did Not Begin With One Chosen Strategy

The final QuantConnect implementation should not be interpreted as a strategy that was invented first and subsequently backtested.

The research process across NB00–NB10 was deliberately broader.

We began by constructing a controlled market environment and then examined several possible ways of extracting information from that environment.

The system considered multiple:

- features,
- predictive models,
- signal-generation mechanisms,
- portfolio-construction methods,
- risk treatments,
- stress assumptions,
- execution assumptions,
- and eventually different combinations of those elements.

The strategy that reaches QuantConnect is therefore the result of a **selection process**.

There were many alternatives.

Some performed better.

Some performed worse.

Some were discarded because they produced weak predictive evidence.

Others could be rejected because their portfolio behavior was unstable, their drawdowns were excessive, their costs were too large, or their performance failed under stress.

The system was intentionally designed to preserve those losing alternatives rather than pretend that the final strategy was obvious from the beginning.

---

# 2. The Basic Research Question

At the investment level, the system ultimately investigates a relatively straightforward hypothesis:

> Can information contained in recent returns, momentum, volatility, trading activity, and trend position help us distinguish relatively stronger equities from relatively weaker equities?

But there is an important second question:

> Which modeling technique is best at extracting that information?

We deliberately did not assume the answer.

Instead, several different model families were allowed to compete.

This is one of the principal places where AI and machine learning entered the process.

---

# 3. Several Competing Models Were Considered

The research architecture included, among others:

- Logistic Regression
- K-Nearest Neighbors
- Random Forest
- Multi-Layer Perceptron / Neural Network
- Linear Regression

These models represent substantially different ways of interpreting exactly the same market evidence.

For example, Logistic Regression assumes a relatively simple relationship between the inputs and the probability of an upward move.

KNN asks whether the present market observation resembles historical observations and then examines what happened after those similar situations.

Random Forest looks for nonlinear thresholds and interactions between the variables.

The neural network can identify still more complex nonlinear relationships.

Linear Regression approaches the problem differently by estimating a continuous future return instead of directly predicting a binary direction.

The important point for the Investment Committee is therefore:

> **The investment strategy was not based on one model that someone subjectively preferred. Multiple competing models were placed into an empirical tournament.**

---

# 4. The Models Were Not Allowed to Select Themselves

Although these are AI and machine-learning models, **the winner was not chosen by an AI opinion**.

This is crucial.

The selection mechanism is deterministic.

The process is approximately:

**Historical observations**

→ **Training period**

→ **Validation period**

→ **Evaluate each candidate model**

→ **Calculate predefined performance metric**

→ **Rank the models**

→ **Select the highest-scoring model**

The principal validation criterion in the QuantConnect implementation is balanced classification accuracy.

Therefore, if the scores were hypothetically:

| Model | Validation Score |
|---|---:|
| Logistic Regression | 0.523 |
| KNN | 0.541 |
| Random Forest | 0.566 |
| Neural Network | 0.548 |
| Linear Regression | 0.517 |

the system would choose:

**Random Forest**

not because anyone thought Random Forest was intellectually more sophisticated, but because it produced the best predefined validation result.

The remaining models would be **discarded for that particular training cycle**.

They are not deleted from the system.

At the next retraining date, they compete again.

A different model may then become champion.

This means the system can adapt.

For example:

**January → Random Forest**

**February → Logistic Regression**

**March → KNN**

**April → Random Forest**

The strategy is therefore not simply an AI model.

It is a **governed model-selection system**.

---

# 5. AI Therefore Participated First as a Search Mechanism

One of the most useful ways to explain this to an Investment Committee is that AI dramatically enlarged the number of hypotheses that could be examined.

Traditional quantitative research might begin with a hypothesis such as:

> Twenty-day momentum should predict future returns.

The researcher might estimate one regression, choose some thresholds, and create a portfolio.

Our architecture instead permits a much larger search space.

For example:

**Momentum information**

could be interpreted by:

- Logistic Regression,
- KNN,
- Random Forest,
- Neural Network,
- Linear Regression.

Those outputs could then potentially be converted into strategies using different rules.

Those strategies could subsequently be combined with different portfolio methods.

Therefore, AI participates partly by allowing the research institution to ask:

> Which combination of model, signal, portfolio, and risk treatment appears most robust?

rather than:

> Does my favorite model work?

That difference is extremely important.

---

# 6. There Were Several Layers of Competition

The alternatives were not limited to the choice of predictive model.

The broader NB00–NB10 architecture considered several layers.

Conceptually:

**DATA**

↓

**FEATURES**

↓

**MODELS**

↓

**SIGNALS**

↓

**PORTFOLIO CONSTRUCTION**

↓

**RISK**

↓

**EXECUTION**

Each layer contains choices.

Therefore, the candidate being evaluated is not merely:

**Random Forest**

It is closer to:

> Random Forest + specific feature set + ranking rule + long/short construction + inverse-volatility weighting + transaction-cost model + risk constraints.

A different candidate might be:

> KNN + same features + same ranking rule + equal weighting.

Another might use a neural network.

Another could use linear regression.

The architecture was built so these alternatives can be compared rather than hidden.

---

# 7. AI Was Used to Extract Information From the Data

The database created at the beginning of the project supplied market information such as:

- prices,
- returns,
- volume,
- corporate events,
- market regimes,
- security metadata,
- and controlled data anomalies.

From these observations, the later notebooks created point-in-time features including:

- one-day returns,
- five-day momentum,
- twenty-day momentum,
- twenty-day volatility,
- sixty-day volatility,
- abnormal volume,
- abnormal dollar volume,
- price relative to moving averages,
- corporate-event information.

The machine-learning models then attempted to determine whether combinations of those variables contained predictive information.

Therefore AI participated in answering questions such as:

> Does momentum matter differently when volatility is high?

> Does unusually high trading activity change the probability associated with a momentum signal?

> Are nonlinear combinations of these variables more informative than linear combinations?

A Random Forest, KNN, or neural network can discover relationships that are difficult to specify manually.

However, this does **not** mean every discovered relationship is accepted.

It must survive out-of-sample validation.

---

# 8. AI Helped Generate Signals, But Did Not Directly Generate Orders

Another important governance distinction is the separation between:

**prediction**

and:

**investment action**.

The AI models produce scores.

For example:

| Security | Model Score |
|---|---:|
| NVDA | +0.24 |
| MSFT | +0.19 |
| JPM | +0.15 |
| ... | ... |
| C | −0.17 |
| UPS | −0.21 |

Those numbers are not orders.

The next deterministic layer ranks the securities.

Approximately:

**top 20% → long candidates**

**middle 60% → no position**

**bottom 20% → short candidates**

Thus AI contributes information.

A separate portfolio engine determines how that information becomes capital allocation.

This is deliberately similar to the structure of a traditional investment institution:

**Research produces views.**

**Portfolio construction converts views into weights.**

**Risk can modify or reject those weights.**

**Execution implements only approved positions.**

---

# 9. Portfolio Construction Was Another Selection Layer

Even after selecting the predictive model, there are several possible ways of constructing the portfolio.

The research architecture contemplated alternatives including:

- Equal Weighting
- Inverse Volatility
- Volatility Targeting
- Risk-Parity-like approaches

The QuantConnect implementation currently uses:

**Inverse Volatility**

as its principal weighting mechanism.

That means lower-volatility positions generally receive larger weights than higher-volatility positions.

Again, the important point is:

> The predictive model does not determine position size.

The investment process remains modular.

---

# 10. Risk Was Independent From AI

The strategy produced by the model-selection process is still not automatically accepted.

It must pass the independent risk layer.

The current QuantConnect implementation includes limits such as:

- maximum 10% position size,
- maximum 35% sector gross exposure,
- maximum 100% total gross exposure,
- 12% portfolio drawdown halt.

The architecture is intentionally:

**AI model**

→ **signal**

→ **portfolio**

→ **independent risk**

→ **execution**

not:

**AI model**

→ **trade**

The distinction is fundamental from an institutional-governance perspective.

AI can produce an investment hypothesis.

AI cannot override risk.

---

# 11. AI Was Also Used to Challenge the Strategy

The role of AI became substantially more sophisticated in NB09 and NB10.

Before those notebooks, most of the "intelligence" was machine learning and deterministic orchestration.

NB09 introduced the first genuine Large Language Model into the architecture.

The LLM was given a bounded role.

Its first function was:

> **Mission interpretation and research-plan proposal.**

For example, a human researcher could ask in ordinary language:

> Evaluate whether an event-aware cross-sectional equity strategy remains robust across sectors, regimes, model families, portfolio methods, and realistic costs.

The LLM could interpret that request and propose:

- which approved model families should be tested,
- which strategy branches should be explored,
- which portfolio approaches should be included,
- which stress tests were relevant.

But that proposal was not executable.

The LLM output became an **intermediate representation**.

A deterministic compiler then checked:

- whether every proposed model existed,
- whether every tool was authorized,
- whether resources were sufficient,
- whether dependencies were valid,
- whether the research remained within scope.

Only after those checks could experiments run.

---

# 12. The LLM Therefore Helped Decide What to Test, Not What to Buy

This is an important distinction for the Investment Committee.

The LLM did not say:

> Buy NVIDIA.

Instead, its role was closer to that of a research strategist:

> Given the research objective and approved capabilities, these are the models and experiments that should be examined.

The quantitative research engine then produced actual empirical evidence.

Therefore:

**LLM → proposes experiments**

**Machine-learning models → produce predictions**

**Validation engine → compares alternatives**

**Portfolio engine → constructs exposures**

**Risk engine → independently constrains exposures**

**Human → retains ultimate authority**

---

# 13. The Second LLM Role Was Research Criticism

After the quantitative work was complete, the LLM returned in a different role.

It was shown compact empirical evidence including:

- model diagnostics,
- leading strategies,
- portfolio results,
- the mathematically selected champion,
- independent risk results,
- stress tests.

Its job was then to behave like an independent research critic.

It could ask questions such as:

> Does the result depend too heavily on one market period?

> Does the performance disappear under higher transaction costs?

> Is the result concentrated in one sector?

> Is the winning model materially better than simpler alternatives?

> Are there reasons to run additional experiments?

This is a particularly useful form of AI participation.

Instead of using AI only to find positive results, AI is also used to **challenge positive results**.

---

# 14. The LLM Was Explicitly Forbidden From Promoting the Strategy

The architecture contains a hard governance rule:

`HUMAN_REVIEW_REQUIRED`

The research critic may:

- explain,
- challenge,
- identify weaknesses,
- recommend experiments,
- summarize evidence.

It may not say:

> This strategy is approved for deployment.

The system's schema itself hard-codes:

`promotion_view = HUMAN_REVIEW_REQUIRED`

That means neither the predictive AI nor the generative AI controls investment authorization.

---

# 15. Alternatives Were Explicitly Discarded

This is perhaps the most important part of the Investment Committee explanation.

The final strategy should be understood as the survivor of a **falsification process**.

At several stages, alternatives can be rejected.

A model can be rejected because:

- validation accuracy is inferior,
- predictions are unstable,
- performance disappears outside the training sample.

A strategy can be rejected because:

- Sharpe ratio deteriorates,
- drawdowns become excessive,
- turnover becomes too high,
- transaction costs eliminate the edge.

A portfolio method can be rejected because:

- concentration becomes excessive,
- volatility is unstable,
- sector exposure becomes unacceptable.

A complete strategy can be rejected because:

- performance exists only in one regime,
- corporate events create instability,
- data defects materially change the result,
- adversarial perturbations destroy the result,
- performance depends excessively on one security or sector,
- independent risk rejects the resulting exposure.

Therefore the research process is not merely:

**find a good strategy**

It is:

> **Generate competing hypotheses and systematically try to reject them.**

The champion is simply the alternative that currently survives best.

---

# 16. A Useful Investment Committee Analogy

The architecture can be compared with a traditional investment organization.

### Traditional organization

**Analysts**

generate investment ideas.

**Portfolio managers**

compare them and determine capital allocations.

**Risk management**

places independent constraints on those allocations.

**Investment Committee**

decides whether the strategy belongs in the institution.

### Our AI architecture

**Machine-learning models**

generate competing predictive views.

**Validation engine**

determines which model currently has the strongest evidence.

**Portfolio engine**

translates rankings into allocations.

**Independent Risk Governor**

constrains exposures.

**LLM research critic**

challenges the evidence and recommends additional tests.

**Human Investment Committee**

retains final authority.

This analogy is useful because it demonstrates that AI did not replace the institution.

It became another component inside the institution.

---

# 17. How the Notebook Progression Fits Together

The progression from NB00 to NB10 can therefore be explained approximately as follows.

## NB00 — Build the controlled investment world

The first notebook established the data environment.

The objective was to create a market substrate sufficiently rich to support controlled research, including:

- multiple securities,
- sectors,
- prices,
- volume,
- regimes,
- corporate actions,
- controlled defects,
- provenance.

AI had not yet become the primary decision mechanism.

The purpose was to create a trustworthy laboratory.

---

## NB01 — Introduce competing models and strategies

This was one of the first major points where machine learning became central.

Several predictive methods were evaluated rather than choosing one ex ante.

The architecture began comparing:

- Logistic Regression,
- KNN,
- neural methods,
- momentum-based logic,
- and other quantitative approaches.

This is where the basic principle emerged:

> **Do not choose a model because we like it. Make models compete.**

---

## NB02 — Add execution and independent risk

The project then recognized that predictive accuracy alone is not an investment strategy.

It introduced:

- portfolio effects,
- execution assumptions,
- risk controls,
- auditing.

This allowed otherwise attractive models to be discarded if their portfolio implementation was unacceptable.

---

## NB03 — Convert capabilities into reusable tools

Models and analytical functions became registered capabilities.

This meant the system no longer had to "remember" how to estimate everything from scratch.

A model became a tool that could be selected when relevant.

This was the beginning of the autonomous-system architecture.

---

## NB04 — Organize tools into governed constellations

Instead of running isolated models, the architecture began combining specialized components.

Different tasks could be assigned to different tools and agents.

The system became capable of constructing larger research workflows.

---

## NB05 — Introduce a meta-agent

The system acquired a higher-level orchestration layer.

Instead of manually deciding every research step, a meta-agent could determine which approved capabilities were relevant to a mission.

This did not yet imply full LLM cognition.

Much of the orchestration remained deterministic.

---

## NB06 — Create a closed-loop research mission

Research became a governed process with:

- mission admission,
- planning,
- budget allocation,
- state transitions,
- dependency checks,
- execution,
- validation,
- provenance,
- audit.

At this stage, the architecture began resembling an autonomous research institution rather than a collection of models.

---

## NB07 — Translate the institution into QuantConnect

NB07 is the implementation bridge.

It converts the research architecture into a LEAN-compatible backtest.

The current QuantConnect implementation preserves:

- model competition,
- champion selection,
- point-in-time features,
- cross-sectional ranking,
- portfolio construction,
- independent risk,
- corporate-action handling,
- costs,
- audit.

The purpose is not to let AI trade autonomously.

The purpose is to verify that the research architecture can survive translation into a real event-driven execution engine.

---

## NB08 — Integrate earlier quantitative and agent capabilities

NB08 combined the earlier model, strategy, portfolio, agent, and governance machinery into a broader unified system.

However, its "understanding" layer was still largely deterministic.

The orchestration could dynamically choose tools, but it was not yet genuinely language-model powered.

---

## NB09 — Introduce the real LLM

NB09 changed the cognitive control plane.

The LLM could now:

- understand an ordinary-language mission,
- propose relevant research branches,
- select from approved model tools,
- identify appropriate experiments,
- critique empirical results.

But crucially:

**the LLM proposal did not directly execute.**

It had to survive deterministic compilation and validation.

---

## NB10 — Integrate the complete research institution

NB10 brought together the full progression.

The system contained:

- the complete data architecture,
- corporate events,
- feature engineering,
- multiple models,
- strategy construction,
- portfolio alternatives,
- execution costs,
- independent risk,
- stress testing,
- adversarial testing,
- LLM planning,
- deterministic compilation,
- LLM criticism,
- provenance,
- audit,
- human authority.

NB10 is therefore best understood not as one AI model but as an **AI-enabled quantitative research institution**.

---

# 18. What Was Actually Selected

The QuantConnect strategy represents one current surviving implementation:

**Universe:** 30 equities

**Feature family:** momentum, volatility, volume, trend

**Candidate models:**

- Logistic Regression
- KNN
- Random Forest
- MLP
- Linear Regression

**Selection mechanism:**

- chronological validation,
- predefined quantitative metric,
- deterministic ranking.

**Signal:**

- rank securities cross-sectionally.

**Portfolio:**

- approximately top 20% long,
- bottom 20% short.

**Weighting:**

- primarily inverse volatility.

**Risk:**

- individual position limit,
- sector concentration limit,
- gross exposure limit,
- portfolio drawdown halt.

Thus the final strategy is not simply:

> "We chose Random Forest."

The better description is:

> **We created an institutional process in which several AI and quantitative models compete repeatedly, the strongest current model generates relative investment views, those views are translated into a portfolio through deterministic rules, and independent risk retains veto authority.**

---

# 19. Where AI Added Value

AI added value in at least four distinct ways.

## First: broader hypothesis search

AI allowed many alternative modeling approaches to be evaluated economically.

Instead of one research hypothesis, the system could compare many.

---

## Second: nonlinear information extraction

Machine-learning models can identify relationships among:

- momentum,
- volatility,
- volume,
- trend,
- market conditions

that may be difficult to specify manually.

---

## Third: adaptive model selection

The system can periodically determine which model currently provides the best empirical evidence rather than assuming that one method remains permanently superior.

---

## Fourth: research orchestration and criticism

The LLM can help determine:

- which experiments deserve attention,
- which model families should be tested,
- what assumptions may be fragile,
- which additional falsification tests are needed.

This moves AI from purely predictive modeling into **research process intelligence**.

---

# 20. What AI Did Not Do

For governance purposes, the Committee should be equally clear about what AI did **not** do.

AI did not:

- authorize live trading,
- override portfolio risk limits,
- override the independent risk function,
- decide that a strategy should be promoted,
- connect autonomously to a broker,
- increase its own investment mandate,
- convert an LLM recommendation directly into an order.

The final authority remains human.

---

# 21. The Correct Committee-Level Interpretation

The most accurate statement is therefore:

> **Artificial intelligence did not replace the investment process. It expanded and strengthened the investment research process.**

We used AI to:

- extract information,
- compare alternative models,
- explore a larger strategy space,
- dynamically propose research experiments,
- select predictive champions through empirical evidence,
- and critically challenge the resulting strategy.

But we deliberately separated those functions from:

- portfolio authority,
- risk authority,
- execution authority,
- and investment approval.

The system therefore embodies a very different philosophy from an unconstrained "AI trader."

It is closer to:

> **an AI-enhanced investment research institution with deterministic portfolio governance and human fiduciary authority.**

---

# 22. The Most Important Point About the Alternatives That Were Discarded

The final strategy is meaningful precisely because there were alternatives that did not survive.

If only one model had been estimated, its performance would tell us relatively little.

By forcing several models and strategies to compete, the system creates a stronger evidentiary process.

The losing alternatives are informative.

They tell us:

- which representations of the data were weaker,
- which models failed to generalize,
- which portfolio structures created undesirable risks,
- which apparent opportunities disappeared after costs,
- which strategies failed stress tests.

Therefore, discarded alternatives are not wasted work.

They are part of the evidence supporting—or challenging—the surviving strategy.

This is analogous to scientific research.

A result becomes more credible when competing explanations have been tested and rejected.

---

# 23. The Investment Committee Should Therefore View the Champion as Provisional

The word **champion** should not be interpreted as:

> permanently approved investment strategy.

It means:

> the currently strongest candidate among the alternatives tested under the current research protocol.

That candidate can later lose.

A different model can become champion.

A stress test can invalidate the current strategy.

Transaction costs can remove its advantage.

A new market regime can reduce its effectiveness.

The independent risk engine can halt it.

The Investment Committee can reject it.

This is intentional.

The architecture is designed so that the system remains capable of changing its mind when evidence changes.

---

# 24. Final Investment Committee Summary

The progression from NB00 through NB10 should therefore be understood as the creation of a **governed AI investment research process**.

We did not ask AI:

> "What should we buy?"

We asked a much more institutional question:

> "Given a controlled body of market information, which models, signals, and portfolio structures survive systematic empirical competition and falsification?"

Artificial intelligence participated in:

**information extraction**

→ **model estimation**

→ **comparison of alternative predictive hypotheses**

→ **research-plan generation**

→ **adaptive model selection**

→ **strategy exploration**

→ **critical review of the evidence**

But actual investment action remained governed by:

**validation rules**

→ **portfolio rules**

→ **independent risk**

→ **execution controls**

→ **human authority**

The key message for the Investment Committee is therefore:

> **AI did not select a security because it liked a story. It helped us systematically examine a large set of alternative quantitative hypotheses. The alternatives were tested against one another, weaker candidates were discarded, the strongest surviving model generated the investment signal, independent risk constrained the resulting portfolio, and the final authority remained with the human investment process.**

Or, in its simplest form:

**AI searched.**

**Models competed.**

**Evidence selected.**

**Risk constrained.**

**Humans remained responsible.**